In [ ]:
!pip install reportlab


In [ ]:
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)


In [ ]:
import os
from google.colab import drive

# Attempt to unmount first, if it was mounted
try:
    drive.flush_and_unmount()
    print("Drive unmounted successfully.")
except ValueError:
    print("Drive was not mounted or could not be unmounted.")

# Clean up the mount point directory contents if any
mount_point = "/content/drive"
if os.path.exists(mount_point) and os.listdir(mount_point):
    print(f"Clearing contents of {mount_point}...")
    # This command recursively deletes content but keeps the directory itself
    !rm -rf {mount_point}/*

# Now mount the drive
print("Attempting to mount Google Drive...")
drive.mount(mount_point, force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Drive unmounted successfully.
Attempting to mount Google Drive...
Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!/usr/bin/env python3
"""
finance_research_pipeline.py
Simplified: **data loading only**.

This module is now responsible purely for:
- defining the ticker universe and sector map
- downloading and caching OHLCV data for all tickers
- saving per-ticker historical CSVs under `finance_output/`

No feature engineering, correlation analysis, or reporting logic is included.
Those should be implemented in separate scripts (e.g. `world_model_correlation_pipeline.py`,
`baseline_vs_worldmodel.py`, `multi_model_benchmark.py`).
"""

import os
import time
import logging
from datetime import datetime
from typing import List, Dict

import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # e.g. running inside a notebook / Colab where __file__ is undefined
    _HERE = os.getcwd()

DATA_DIR = os.path.join(_HERE, "finance_output")
os.makedirs(DATA_DIR, exist_ok=True)


# ────────────────────────────────────────────────────────────────────────────────
# 1. Ticker universe and sector map
# ────────────────────────────────────────────────────────────────────────────────

TICKER_UNIVERSE: Dict[str, List[str]] = {
    "Technology": [
        "AAPL","MSFT","NVDA","GOOGL","META","TSLA","ADBE","CRM","ORCL","INTC",
        "CSCO","IBM","QCOM","TXN","AVGO","AMD","MU","AMAT","LRCX","KLAC",
        "SNPS","CDNS","FTNT","PANW","CRWD","ZS","DDOG","SNOW","PLTR","NET",
        "SHOP","SQ","PYPL","WDAY","VEEV","TEAM","DOCU","MDB","ANET","SMCI",
        "HPQ","DELL","STX","WDC","NTAP","PSTG","OKTA","ZM","TWLO","BILL",
    ],
    "Financials": [
        "JPM","BAC","WFC","GS","MS","C","AXP","V","MA","COF",
        "BLK","SCHW","USB","PNC","TFC","FITB","KEY","RF","HBAN","CFG",
        "BK","STT","NTRS","TROW","IVZ","PRU","MET","AFL","ALL","TRV",
        "HIG","CB","AON","MMC","ICE","CME","CBOE","NDAQ","HOOD","SOFI",
    ],
    "Healthcare": [
        "JNJ","UNH","ABT","TMO","PFE","MRK","LLY","BMY","GILD","AMGN",
        "BIIB","REGN","VRTX","MRNA","BSX","SYK","MDT","BAX","BDX","ISRG",
        "IDXX","IQV","CRL","ILMN","ALGN","HOLX","HSIC","CVS","CI","HUM",
    ],
    "Consumer_Discretionary": [
        "AMZN","HD","MCD","NKE","SBUX","TGT","LOW","TJX","ROST","BKNG",
        "MAR","HLT","CCL","RCL","MGM","WYNN","ABNB","UBER","DASH","EXPE",
        "AZO","ORLY","F","GM","RIVN","NIO","RL","BBY","ETSY","EBAY",
    ],
    "Consumer_Staples": [
        "WMT","PG","KO","PEP","COST","PM","MO","MDLZ","KHC","GIS",
        "K","CPB","HRL","MKC","CAG","COKE","KDP","STZ","MNST","CELH",
    ],
    "Industrials": [
        "CAT","DE","HON","GE","BA","RTX","LMT","NOC","GD","MMM",
        "EMR","ETN","ROK","PH","DOV","ITW","AME","ROP","FDX","UPS",
        "CHRW","EXPD","JBHT","KNX","URI","RSG","WM","UNP","CSX","NSC",
    ],
    "Energy": [
        "XOM","CVX","COP","EOG","PXD","DVN","MPC","VLO","PSX","SLB",
        "HAL","BKR","OXY","APA","MRO","HES","EQT","AR","RRC","CTRA",
        "ET","EPD","OKE","WMB","LNG","NEE","AES","SO","DUK","PCG",
    ],
    "Materials": [
        "LIN","APD","SHW","ECL","PPG","RPM","DOW","DD","LYB","NEM",
        "GOLD","AEM","FCX","AA","NUE","STLD","CMC","RS","ATI","HWM",
    ],
    "Real_Estate": [
        "AMT","PLD","CCI","EQIX","PSA","DLR","O","WELL","AVB","EQR",
        "MAA","UDR","CPT","NNN","ADC","STAG","IIPR","VNO","BXP","SPG",
    ],
    "Communication_Services": [
        "GOOG","META","NFLX","DIS","CMCSA","T","VZ","TMUS","CHTR",
        "SNAP","PINS","RDDT","BMBL","MTCH","TTWO","EA","RBLX","SPOT","NYT","AMCX",
    ],
    "Utilities": [
        "NEE","DUK","SO","D","AEP","EXC","SRE","PCG","PEG","ED",
        "XEL","ES","EIX","DTE","PPL","AEE","LNT","WEC","AWK","WTRG",
    ],
    "ETFs_Indices": [
        "SPY","QQQ","IWM","DIA","VTI","VOO","GLD","SLV","TLT","IEF",
        "AGG","BND","LQD","HYG","XLF","XLK","XLV","XLE","XLI","VNQ",
    ],
}

ALL_TICKERS: List[str] = []
TICKER_SECTOR: Dict[str, str] = {}
for sector, tickers in TICKER_UNIVERSE.items():
    for t in tickers:
        if t not in TICKER_SECTOR:
            ALL_TICKERS.append(t)
            TICKER_SECTOR[t] = sector


# ────────────────────────────────────────────────────────────────────────────────
# 2. Data loading helpers
# ────────────────────────────────────────────────────────────────────────────────

def download_ohlcv_for_ticker(ticker: str, start: str = "2018-01-01", end: str = None) -> pd.DataFrame:
    """
    Download daily OHLCV data for one ticker from yfinance.
    Returns a DataFrame indexed by date with columns: Open, High, Low, Close, Adj Close, Volume.
    """
    end = end or datetime.today().strftime("%Y-%m-%d")
    print(f"  [{ticker}] downloading OHLCV {start} → {end} ...")
    data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if data is None or data.empty:
        raise RuntimeError(f"No OHLCV data returned for {ticker}")
    data = data.reset_index().set_index("Date").sort_index()
    return data


def save_ohlcv_csv(ticker: str, df: pd.DataFrame) -> str:
    """Save one ticker's OHLCV DataFrame to `finance_output/{ticker}_historical.csv`."""
    out_path = os.path.join(DATA_DIR, f"{ticker}_historical.csv")
    df.to_csv(out_path)
    return out_path


def download_all_tickers(
    tickers: List[str] = None,
    start: str = "2018-01-01",
    end: str = None,
    sleep_sec: float = 0.5,
) -> None:
    """
    Download and cache OHLCV for a list of tickers.

    - Uses yfinance (free, public) data.
    - Writes one CSV per ticker into `finance_output/`.
    - Skips tickers where a non-empty CSV already exists.
    """
    if tickers is None:
        tickers = ALL_TICKERS

    end = end or datetime.today().strftime("%Y-%m-%d")
    print("═" * 72)
    print("Finance data loader — OHLCV download")
    print("═" * 72)
    print(f"Tickers: {len(tickers)}  |  Period: {start} → {end}")
    print(f"Output directory: {DATA_DIR}")
    print("─" * 72)

    ok, failed = [], []
    for i, sym in enumerate(tickers, 1):
        out_path = os.path.join(DATA_DIR, f"{sym}_historical.csv")
        if os.path.isfile(out_path):
            try:
                if not pd.read_csv(out_path).empty:
                    print(f"[{i:4d}/{len(tickers)}] {sym:8s} already cached, skipping.")
                    continue
            except Exception:
                pass
        try:
            df = download_ohlcv_for_ticker(sym, start=start, end=end)
            save_ohlcv_csv(sym, df)
            ok.append(sym)
            print(f"[{i:4d}/{len(tickers)}] {sym:8s} saved → {out_path}")
        except Exception as e:
            failed.append(sym)
            print(f"[{i:4d}/{len(tickers)}] {sym:8s} FAILED: {e}")
        time.sleep(sleep_sec)

    print("─" * 72)
    print(f"Completed. Success: {len(ok)}, Failed: {len(failed)}")
    if failed:
        print("Failed tickers:", ", ".join(failed))


if __name__ == "__main__":
    # Simple CLI entrypoint: download all tickers in the universe.
    download_all_tickers()

#!/usr/bin/env python3
"""
finance_research_pipeline.py
═══════════════════════════════════════════════════════════════════════════════
Finance World Model — Standalone Research Pipeline

Run:
    python finance_research_pipeline.py

What it does:
    1. Asks how many companies to analyse (or specific tickers)
    2. Fetches OHLCV + financials from yfinance (parallel, auto-retry, resume)
    3. Engineers 91 features per ticker (9 families)
    4. Preprocesses: missing audit, stationarity, distribution, winsorise, scale
    5. Correlation analysis: Pearson, Spearman, feature→target, clustermap,
       rolling stability, family-level grouped bars
    6. Generates a publication-quality ReportLab PDF research report

Requirements:
    pip install yfinance reportlab statsmodels scikit-learn seaborn scipy pandas numpy matplotlib
"""

# ── stdlib ────────────────────────────────────────────────────────────────────
import os, sys, time, warnings, logging, threading, textwrap, io
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import RobustScaler
import yfinance as yf

# ReportLab
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm, mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, Image, HRFlowable, KeepTogether
)
from reportlab.platypus.flowables import BalancedColumns
from reportlab.graphics.shapes import Drawing, Line, Rect, String
from reportlab.pdfbase import pdfmetrics

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', message='.*p-value.*look-up table.*', module='statsmodels.*')
try:
    from statsmodels.tools.sm_exceptions import InterpolationWarning as _IW
    warnings.filterwarnings('ignore', category=_IW)
except Exception:
    pass
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)

# ═══════════════════════════════════════════════════════════════════════════════
# 0 ── COLOUR PALETTE & STYLES
# ═══════════════════════════════════════════════════════════════════════════════

# Report colours
R_NAVY   = colors.HexColor('#0D2B4E')
R_BLUE   = colors.HexColor('#185FA5')
R_TEAL   = colors.HexColor('#0F6E56')
R_AMBER  = colors.HexColor('#BA7517')
R_CORAL  = colors.HexColor('#993C1D')
R_GRAY   = colors.HexColor('#5F5E5A')
R_LGRAY  = colors.HexColor('#D3D1C7')
R_BG     = colors.HexColor('#F8F8F6')
R_WHITE  = colors.white
R_BLACK  = colors.HexColor('#1A1A18')

# Plot colours
P_BLUE  = '#185FA5'; P_TEAL  = '#0F6E56'; P_AMBER = '#BA7517'
P_CORAL = '#993C1D'; P_GRAY  = '#5F5E5A'; P_BG    = '#F8F8F6'

sns.set_theme(style='whitegrid', font_scale=0.85)
plt.rcParams.update({'figure.facecolor': P_BG, 'axes.facecolor': P_BG,
                     'savefig.facecolor': P_BG, 'savefig.dpi': 150,
                     'font.family': 'DejaVu Sans'})

# ═══════════════════════════════════════════════════════════════════════════════
# 1 ── TICKER UNIVERSE
# ═══════════════════════════════════════════════════════════════════════════════

TICKER_UNIVERSE = {
    "Technology":             ["AAPL","MSFT","NVDA","GOOGL","META","TSLA","ADBE","CRM","ORCL","INTC",
                                "CSCO","IBM","QCOM","TXN","AVGO","AMD","MU","AMAT","LRCX","KLAC",
                                "SNPS","CDNS","FTNT","PANW","CRWD","ZS","DDOG","SNOW","PLTR","NET",
                                "SHOP","SQ","PYPL","WDAY","VEEV","TEAM","DOCU","MDB","ANET","SMCI",
                                "HPQ","DELL","STX","WDC","NTAP","PSTG","OKTA","ZM","TWLO","BILL"],
    "Financials":             ["JPM","BAC","WFC","GS","MS","C","AXP","V","MA","COF",
                                "BLK","SCHW","USB","PNC","TFC","FITB","KEY","RF","HBAN","CFG",
                                "BK","STT","NTRS","TROW","IVZ","PRU","MET","AFL","ALL","TRV",
                                "HIG","CB","AON","MMC","ICE","CME","CBOE","NDAQ","HOOD","SOFI"],
    "Healthcare":             ["JNJ","UNH","ABT","TMO","PFE","MRK","LLY","BMY","GILD","AMGN",
                                "BIIB","REGN","VRTX","MRNA","BSX","SYK","MDT","BAX","BDX","ISRG",
                                "IDXX","IQV","CRL","ILMN","ALGN","HOLX","HSIC","CVS","CI","HUM"],
    "Consumer_Discretionary": ["AMZN","HD","MCD","NKE","SBUX","TGT","LOW","TJX","ROST","BKNG",
                                "MAR","HLT","CCL","RCL","MGM","WYNN","ABNB","UBER","DASH","EXPE",
                                "AZO","ORLY","F","GM","RIVN","NIO","RL","BBY","ETSY","EBAY"],
    "Consumer_Staples":       ["WMT","PG","KO","PEP","COST","PM","MO","MDLZ","KHC","GIS",
                                "K","CPB","HRL","MKC","CAG","COKE","KDP","STZ","MNST","CELH"],
    "Industrials":            ["CAT","DE","HON","GE","BA","RTX","LMT","NOC","GD","MMM",
                                "EMR","ETN","ROK","PH","DOV","ITW","AME","ROP","FDX","UPS",
                                "CHRW","EXPD","JBHT","KNX","URI","RSG","WM","UNP","CSX","NSC"],
    "Energy":                 ["XOM","CVX","COP","EOG","PXD","DVN","MPC","VLO","PSX","SLB",
                                "HAL","BKR","OXY","APA","MRO","HES","EQT","AR","RRC","CTRA",
                                "ET","EPD","OKE","WMB","LNG","NEE","AES","SO","DUK","PCG"],
    "Materials":              ["LIN","APD","SHW","ECL","PPG","RPM","DOW","DD","LYB","NEM",
                                "GOLD","AEM","FCX","AA","NUE","STLD","CMC","RS","ATI","HWM"],
    "Real_Estate":            ["AMT","PLD","CCI","EQIX","PSA","DLR","O","WELL","AVB","EQR",
                                "MAA","UDR","CPT","NNN","ADC","STAG","IIPR","VNO","BXP","SPG"],
    "Communication_Services": ["GOOG","META","NFLX","DIS","CMCSA","T","VZ","TMUS","CHTR",
                                "SNAP","PINS","RDDT","BMBL","MTCH","TTWO","EA","RBLX","SPOT","NYT","AMCX"],
    "Utilities":              ["NEE","DUK","SO","D","AEP","EXC","SRE","PCG","PEG","ED",
                                "XEL","ES","EIX","DTE","PPL","AEE","LNT","WEC","AWK","WTRG"],
    "ETFs_Indices":           ["SPY","QQQ","IWM","DIA","VTI","VOO","GLD","SLV","TLT","IEF",
                                "AGG","BND","LQD","HYG","XLF","XLK","XLV","XLE","XLI","VNQ"],
}

════════════════════════════════════════════════════════════════════════
Finance data loader — OHLCV download
════════════════════════════════════════════════════════════════════════
Tickers: 325  |  Period: 2018-01-01 → 2026-05-19
Output directory: /content/finance_output
────────────────────────────────────────────────────────────────────────
  [AAPL] downloading OHLCV 2018-01-01 → 2026-05-19 ...
[   1/325] AAPL     saved → /content/finance_output/AAPL_historical.csv
  [MSFT] downloading OHLCV 2018-01-01 → 2026-05-19 ...
[   2/325] MSFT     saved → /content/finance_output/MSFT_historical.csv
  [NVDA] downloading OHLCV 2018-01-01 → 2026-05-19 ...
[   3/325] NVDA     saved → /content/finance_output/NVDA_historical.csv
  [GOOGL] downloading OHLCV 2018-01-01 → 2026-05-19 ...
[   4/325] GOOGL    saved → /content/finance_output/GOOGL_historical.csv
  [META] downloading OHLCV 2018-01-01 → 2026-05-19 ...
[   5/325] META     saved → /content/finance_output/META_historical.csv
  [TSLA] downloa

In [ ]:
# ============================================================
# Summary Statistics
summary_cols = ["Open", "High", "Low", "Close", "Volume"]

for col in summary_cols:
    combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce")

summary_stats = combined_df[summary_cols].agg(
    ["min", "max", "std"]
).T

summary_stats = summary_stats.round(2)

print(summary_stats)

         min           max          std
Open    0.69  4.334810e+03       181.30
High    0.75  4.388110e+03       183.27
Low     0.64  4.316680e+03       179.31
Close   0.67  4.354540e+03       181.37
Volume  0.00  2.511528e+09  32080133.56


In [ ]:
# ============================================================
# CELL 1 — Install dependencies (run this first, once)
# ============================================================
# !pip install yfinance reportlab statsmodels scikit-learn seaborn scipy pandas numpy matplotlib -q

# ============================================================
# CELL 2 — Full Finance Research Pipeline (paste everything below)
# ============================================================

#!/usr/bin/env python3
"""
finance_research_pipeline.py
═══════════════════════════════════════════════════════════════════════════════
Finance World Model — Standalone Research Pipeline

Run:
    python finance_research_pipeline.py

What it does:
    1. Asks how many companies to analyse (or specific tickers)
    2. Fetches OHLCV + financials from yfinance (parallel, auto-retry, resume)
    3. Engineers 91 features per ticker (9 families)
    4. Preprocesses: missing audit, stationarity, distribution, winsorise, scale
    5. Correlation analysis: Pearson, Spearman, feature→target, clustermap,
       rolling stability, family-level grouped bars
    6. Generates a publication-quality ReportLab PDF research report

Requirements:
    pip install yfinance reportlab statsmodels scikit-learn seaborn scipy pandas numpy matplotlib
"""

# ── stdlib ────────────────────────────────────────────────────────────────────
import os, sys, time, warnings, logging, threading, textwrap, io
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import RobustScaler
import yfinance as yf

# ReportLab
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm, mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, Image, HRFlowable, KeepTogether
)
from reportlab.platypus.flowables import BalancedColumns
from reportlab.graphics.shapes import Drawing, Line, Rect, String
from reportlab.pdfbase import pdfmetrics

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', message='.*p-value.*look-up table.*', module='statsmodels.*')
try:
    from statsmodels.tools.sm_exceptions import InterpolationWarning as _IW
    warnings.filterwarnings('ignore', category=_IW)
except Exception:
    pass
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)

# ═══════════════════════════════════════════════════════════════════════════════
# 0 ── COLOUR PALETTE & STYLES
# ═══════════════════════════════════════════════════════════════════════════════

R_NAVY   = colors.HexColor('#0D2B4E')
R_BLUE   = colors.HexColor('#185FA5')
R_TEAL   = colors.HexColor('#0F6E56')
R_AMBER  = colors.HexColor('#BA7517')
R_CORAL  = colors.HexColor('#993C1D')
R_GRAY   = colors.HexColor('#5F5E5A')
R_LGRAY  = colors.HexColor('#D3D1C7')
R_BG     = colors.HexColor('#F8F8F6')
R_WHITE  = colors.white
R_BLACK  = colors.HexColor('#1A1A18')

P_BLUE  = '#185FA5'; P_TEAL  = '#0F6E56'; P_AMBER = '#BA7517'
P_CORAL = '#993C1D'; P_GRAY  = '#5F5E5A'; P_BG    = '#F8F8F6'

sns.set_theme(style='whitegrid', font_scale=0.85)
plt.rcParams.update({'figure.facecolor': P_BG, 'axes.facecolor': P_BG,
                     'savefig.facecolor': P_BG, 'savefig.dpi': 150,
                     'font.family': 'DejaVu Sans'})

# ═══════════════════════════════════════════════════════════════════════════════
# 1 ── TICKER UNIVERSE
# ═══════════════════════════════════════════════════════════════════════════════

TICKER_UNIVERSE = {
    "Technology": [
        "AAPL","MSFT","NVDA","GOOGL","META","TSLA","ADBE","CRM","ORCL","INTC",
        "CSCO","IBM","QCOM","TXN","AVGO","AMD","MU","AMAT","LRCX","KLAC",
        "SNPS","CDNS","FTNT","PANW","CRWD","ZS","DDOG","SNOW","PLTR","NET",
        "SHOP","SQ","PYPL","WDAY","VEEV","TEAM","DOCU","MDB","ANET","SMCI",
        "HPQ","DELL","STX","WDC","NTAP","PSTG","OKTA","ZM","TWLO","BILL",
    ],
    "Financials": [
        "JPM","BAC","WFC","GS","MS","C","AXP","V","MA","COF",
        "BLK","SCHW","USB","PNC","TFC","FITB","KEY","RF","HBAN","CFG",
        "BK","STT","NTRS","TROW","IVZ","PRU","MET","AFL","ALL","TRV",
        "HIG","CB","AON","MMC","ICE","CME","CBOE","NDAQ","HOOD","SOFI",
    ],
    "Healthcare": [
        "JNJ","UNH","ABT","TMO","PFE","MRK","LLY","BMY","GILD","AMGN",
        "BIIB","REGN","VRTX","MRNA","BSX","SYK","MDT","BAX","BDX","ISRG",
        "IDXX","IQV","CRL","ILMN","ALGN","HOLX","HSIC","CVS","CI","HUM",
    ],
    "Consumer_Discretionary": [
        "AMZN","HD","MCD","NKE","SBUX","TGT","LOW","TJX","ROST","BKNG",
        "MAR","HLT","CCL","RCL","MGM","WYNN","ABNB","UBER","DASH","EXPE",
        "AZO","ORLY","F","GM","RIVN","NIO","RL","BBY","ETSY","EBAY",
    ],
    "Consumer_Staples": [
        "WMT","PG","KO","PEP","COST","PM","MO","MDLZ","KHC","GIS",
        "K","CPB","HRL","MKC","CAG","COKE","KDP","STZ","MNST","CELH",
    ],
    "Industrials": [
        "CAT","DE","HON","GE","BA","RTX","LMT","NOC","GD","MMM",
        "EMR","ETN","ROK","PH","DOV","ITW","AME","ROP","FDX","UPS",
        "CHRW","EXPD","JBHT","KNX","URI","RSG","WM","UNP","CSX","NSC",
    ],
    "Energy": [
        "XOM","CVX","COP","EOG","PXD","DVN","MPC","VLO","PSX","SLB",
        "HAL","BKR","OXY","APA","MRO","HES","EQT","AR","RRC","CTRA",
        "ET","EPD","OKE","WMB","LNG","NEE","AES","SO","DUK","PCG",
    ],
    "Materials": [
        "LIN","APD","SHW","ECL","PPG","RPM","DOW","DD","LYB","NEM",
        "GOLD","AEM","FCX","AA","NUE","STLD","CMC","RS","ATI","HWM",
    ],
    "Real_Estate": [
        "AMT","PLD","CCI","EQIX","PSA","DLR","O","WELL","AVB","EQR",
        "MAA","UDR","CPT","NNN","ADC","STAG","IIPR","VNO","BXP","SPG",
    ],
    "Communication_Services": [
        "GOOG","META","NFLX","DIS","CMCSA","T","VZ","TMUS","CHTR",
        "SNAP","PINS","RDDT","BMBL","MTCH","TTWO","EA","RBLX","SPOT","NYT","AMCX",
    ],
    "Utilities": [
        "NEE","DUK","SO","D","AEP","EXC","SRE","PCG","PEG","ED",
        "XEL","ES","EIX","DTE","PPL","AEE","LNT","WEC","AWK","WTRG",
    ],
    "ETFs_Indices": [
        "SPY","QQQ","IWM","DIA","VTI","VOO","GLD","SLV","TLT","IEF",
        "AGG","BND","LQD","HYG","XLF","XLK","XLV","XLE","XLI","VNQ",
    ],
}

ALL_TICKERS   = []
TICKER_SECTOR = {}
for _s, _tl in TICKER_UNIVERSE.items():
    for _t in _tl:
        if _t not in TICKER_SECTOR:
            ALL_TICKERS.append(_t)
            TICKER_SECTOR[_t] = _s

# ═══════════════════════════════════════════════════════════════════════════════
# 2 ── USER INPUT
# ═══════════════════════════════════════════════════════════════════════════════

def safe_input(prompt: str, default: str = "") -> str:
    """Read from stdin; on EOF use default so script can run non-interactively."""
    try:
        s = input(prompt).strip()
        return s if s else default
    except (EOFError, KeyboardInterrupt):
        return default

def get_user_selection() -> list:
    """Interactive CLI to choose which companies to analyse."""
    print("\n" + "═"*65)
    print("  FINANCE RESEARCH PIPELINE — Company Selection")
    print("═"*65)
    print(f"\n  Total available tickers : {len(ALL_TICKERS)}")
    print(f"  Sectors                 : {len(TICKER_UNIVERSE)}\n")

    for i, (sec, tl) in enumerate(TICKER_UNIVERSE.items(), 1):
        print(f"  {i:2d}. {sec:28s}  ({len(tl)} tickers)")

    print("\n" + "─"*65)
    print("  Options:")
    print("    [1]  Analyse ALL companies")
    print("    [2]  Choose by NUMBER  (e.g. top 50 by sector coverage)")
    print("    [3]  Choose specific SECTORS")
    print("    [4]  Enter custom TICKER LIST")
    print("─"*65)

    choice = safe_input("\n  Enter option [1/2/3/4]: ", "2").strip() or "2"

    if choice == "1":
        print(f"\n  Selected: ALL {len(ALL_TICKERS)} tickers")
        return ALL_TICKERS

    elif choice == "2":
        n_str = safe_input(f"  How many companies? (1–{len(ALL_TICKERS)}): ", "5").strip()
        try:
            n = max(1, min(int(n_str), len(ALL_TICKERS)))
        except ValueError:
            print("  Invalid — defaulting to 5")
            n = 5
        selected, sector_iters = [], {s: iter(t) for s, t in TICKER_UNIVERSE.items()}
        sectors = list(TICKER_UNIVERSE.keys())
        idx = 0
        while len(selected) < n:
            s = sectors[idx % len(sectors)]
            try:
                t = next(sector_iters[s])
                if t not in selected:
                    selected.append(t)
            except StopIteration:
                pass
            idx += 1
            if idx > n * len(sectors) * 2:
                break
        print(f"\n  Selected {len(selected)} tickers (balanced across sectors):")
        print(f"  {', '.join(selected)}")
        return selected

    elif choice == "3":
        print("\n  Available sectors:")
        for i, s in enumerate(TICKER_UNIVERSE.keys(), 1):
            print(f"    {i}. {s}")
        raw = safe_input("  Enter sector numbers (comma-separated, e.g. 1,3,5): ", "1")
        idxs = []
        for x in raw.split(","):
            try: idxs.append(int(x.strip()) - 1)
            except: pass
        sector_names = list(TICKER_UNIVERSE.keys())
        selected = []
        for i in idxs:
            if 0 <= i < len(sector_names):
                selected.extend(TICKER_UNIVERSE[sector_names[i]])
        selected = list(dict.fromkeys(selected))
        print(f"\n  Selected {len(selected)} tickers from chosen sectors")
        return selected

    elif choice == "4":
        raw = safe_input("  Enter tickers (comma or space-separated): ", "AAPL,MSFT")
        selected = [t.strip().upper() for t in raw.replace(",", " ").split() if t.strip()]
        print(f"\n  Selected: {selected}")
        return selected

    else:
        print("  Invalid choice — defaulting to top 10 tickers")
        return ALL_TICKERS[:10]


# ═══════════════════════════════════════════════════════════════════════════════
# 3 ── FETCH
# ═══════════════════════════════════════════════════════════════════════════════

DATASETS = ["historical","info","balance_sheet","income_statement",
            "cash_flow","recommendations","dividends","splits"]

RESET="\033[0m"; BOLD="\033[1m"; DIM="\033[2m"
GREEN="\033[92m"; RED="\033[91m"; YELLOW="\033[93m"

class ProgressTracker:
    def __init__(self, total):
        self.total=total; self.done=0
        self.successful=[]; self.failed=[]; self.partial=[]
        self._lock=threading.Lock(); self._start=time.time()

    def finish(self, sym, status):
        with self._lock:
            self.done += 1
            r = status.get("result","failed")
            if r=="successful": self.successful.append(sym)
            elif r=="failed":   self.failed.append(sym)
            else:               self.partial.append(sym)
            self._print(sym, status)

    def _eta(self):
        if not self.done: return "–"
        rem = (self.total-self.done)/max(self.done/(time.time()-self._start),1e-9)
        m,s=divmod(int(rem),60); return f"{m}m{s:02d}s"

    def _bar(self,w=20):
        f=int(w*self.done/max(self.total,1))
        return f"{GREEN}{'█'*f}{DIM}{'░'*(w-f)}{RESET}"

    def _print(self, sym, status):
        ds=status.get("datasets",{})
        icons="".join(
            f"{GREEN}✓{RESET}" if ds.get(d) is True
            else f"{RED}✗{RESET}" if ds.get(d) is False
            else f"{DIM}–{RESET}" for d in DATASETS)
        res=status.get("result","failed")
        rc=GREEN if res=="successful" else (RED if res=="failed" else YELLOW)
        pct=100*self.done/self.total
        sec=TICKER_SECTOR.get(sym,"")[:11]
        print(f"\r{BOLD}{sym:7s}{RESET} {sec:11s} │ {icons} │ "
              f"{rc}{res[:4].upper()}{RESET} │ [{self._bar()}] "
              f"{pct:5.1f}% {self.done}/{self.total} ETA:{self._eta()}", flush=True)

    def print_header(self):
        ds_h=" ".join(d[:3].upper() for d in DATASETS)
        print(f"\n{BOLD}{'SYM':7s} {'SECTOR':11s} │ {ds_h} │ RES  │ PROGRESS{RESET}")
        print("─"*90)

    def print_summary(self):
        el=time.time()-self._start; m,s=divmod(int(el),60)
        print("\n"+"═"*60)
        print(f"  {GREEN}Successful{RESET}: {len(self.successful):4d}  "
              f"{YELLOW}Partial{RESET}: {len(self.partial):4d}  "
              f"{RED}Failed{RESET}: {len(self.failed):4d}  "
              f"Time: {m}m{s:02d}s")
        if self.failed:
            print(f"  {RED}Failed:{RESET} {', '.join(self.failed[:20])}")
        print("═"*60)


def _fetch_one(sym, out_dir, skip, tracker, retries=3, delay=0.15):
    status={"symbol":sym,"datasets":{}}
    for attempt in range(retries):
        try:
            tk=yf.Ticker(sym)
            def save(name, getter, fname):
                path=f"{out_dir}/{sym}_{fname}.csv"
                if skip and os.path.exists(path):
                    status["datasets"][name]="skip"; return
                try:
                    d=getter()
                    ok=(d is not None and (isinstance(d,dict) and len(d)>0
                        or hasattr(d,'empty') and not d.empty))
                    if ok:
                        (pd.DataFrame.from_dict(d,orient="index",columns=["Value"])
                         if isinstance(d,dict) else d).to_csv(path)
                        status["datasets"][name]=True
                    else: status["datasets"][name]=False
                except: status["datasets"][name]=False

            save("historical",      lambda: tk.history(period="25y",interval="1d"), "historical")
            save("info",            lambda: tk.info,                                 "info")
            save("balance_sheet",   lambda: tk.balance_sheet,                        "balance_sheet")
            save("income_statement",lambda: tk.income_stmt,                          "income_statement")
            save("cash_flow",       lambda: tk.cashflow,                             "cash_flow")
            save("recommendations", lambda: tk.recommendations,                      "recommendations")
            save("dividends",       lambda: tk.dividends,                            "dividends")
            save("splits",          lambda: tk.splits,                               "splits")

            vals=[v for v in status["datasets"].values() if v!="skip"]
            n_ok=sum(1 for v in vals if v is True)
            n_fail=sum(1 for v in vals if v is False)
            status["result"]=("failed" if n_ok==0 and n_fail>0
                              else "partial" if n_fail>0 else "successful")
            import random as _r
            time.sleep(delay + _r.uniform(0, delay * 0.5))
            tracker.finish(sym,status); return status
        except Exception as e:
            if attempt<retries-1: time.sleep(2**attempt); continue
            status.update({"result":"failed","error":str(e)})
    tracker.finish(sym,status); return status


def fetch_all(tickers, out_dir, max_workers=6, delay=0.5, retries=3, skip=True):
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{'═'*60}")
    print(f"  STAGE 1 — Fetching {len(tickers)} tickers")
    print(f"  Workers:{max_workers}  Resume:{'ON' if skip else 'OFF'}  Dir:{out_dir}")
    print(f"{'═'*60}")
    tracker=ProgressTracker(len(tickers))
    tracker.print_header()
    results=[]
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures={ex.submit(_fetch_one,t,out_dir,skip,tracker,retries,delay):t
                 for t in tickers}
        for f in as_completed(futures):
            try: results.append(f.result())
            except Exception as e:
                results.append({"symbol":futures[f],"result":"failed","error":str(e)})
    tracker.print_summary()
    return results, tracker


# ═══════════════════════════════════════════════════════════════════════════════
# 4 ── FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════════

def _ema(s,n): return s.ewm(span=n,adjust=False).mean()
def _sma(s,n): return s.rolling(n).mean()
def _tr(df):
    hl=df['High']-df['Low']
    hcp=(df['High']-df['Close'].shift(1)).abs()
    lcp=(df['Low'] -df['Close'].shift(1)).abs()
    return pd.concat([hl,hcp,lcp],axis=1).max(axis=1)

def _strip_tz(idx):
    """Convert any tz-aware DatetimeIndex to tz-naive UTC wall-clock times."""
    try:
        if hasattr(idx, 'tz') and idx.tz is not None:
            return idx.tz_localize(None)
    except TypeError:
        pass
    try:
        return pd.to_datetime(idx, utc=True).tz_convert(None)
    except Exception:
        return pd.DatetimeIndex(pd.to_datetime(idx, errors='coerce').tz_localize(None)
                                if pd.to_datetime(idx, errors='coerce').tz is None
                                else pd.to_datetime(idx, errors='coerce').tz_convert(None))


def engineer_features(raw, direction_threshold=0.005):
    raw = raw.copy()
    if hasattr(raw.index, 'tz') and raw.index.tz is not None:
        raw.index = raw.index.tz_localize(None)
    raw.index = pd.DatetimeIndex(raw.index)
    frames=[raw.copy()]
    c,h,l,o,v=raw['Close'],raw['High'],raw['Low'],raw['Open'],raw['Volume']
    # F1 Returns
    ret=np.log(c/c.shift(1))
    frames.append(pd.DataFrame({'Ret1':c.pct_change(),'LogRet1':ret,
        'LogRet2':np.log(c/c.shift(2)),'LogRet5':np.log(c/c.shift(5)),
        'LogRet10':np.log(c/c.shift(10)),'LogRet20':np.log(c/c.shift(20)),
        'GapRet':np.log(o/c.shift(1)),'IntradayRet':np.log(c/o),
        'Cum5Ret':c.pct_change(5)},index=raw.index))
    # F2 Trend
    ma={}
    for n in [5,10,20,50,200]: ma[f'SMA{n}']=_sma(c,n); ma[f'EMA{n}']=_ema(c,n)
    e10=_ema(c,10); e20=_ema(c,20); e12=_ema(c,12)
    ma['DEMA10']=2*e10-_ema(e10,10); ma['DEMA20']=2*e20-_ema(e20,20)
    ma['TEMA12']=3*e12-3*_ema(e12,12)+_ema(_ema(e12,12),12)
    w10=np.arange(1,11,dtype=float); w20=np.arange(1,21,dtype=float)
    ma['WMA10']=c.rolling(10).apply(lambda x:np.dot(x,w10)/w10.sum(),raw=True)
    ma['WMA20']=c.rolling(20).apply(lambda x:np.dot(x,w20)/w20.sum(),raw=True)
    ma['VWMA20']=(c*v).rolling(20).sum()/v.rolling(20).sum()
    dist={}
    for n in [10,20,50,200]:
        dist[f'Dist_SMA{n}']=(c-ma[f'SMA{n}'])/c
        dist[f'Dist_EMA{n}']=(c-ma[f'EMA{n}'])/c
    dist.update({'Cross_EMA5_EMA20':(ma['EMA5']-ma['EMA20'])/c,
                 'Cross_EMA10_EMA50':(ma['EMA10']-ma['EMA50'])/c,
                 'Cross_SMA50_SMA200':(ma['SMA50']-ma['SMA200'])/c,
                 'Dist_VWMA20':(c-ma['VWMA20'])/c})
    frames.append(pd.DataFrame(dist,index=raw.index))
    # F3 Momentum
    def rsi(s,n=14):
        d=s.diff(); g=d.clip(lower=0); ls=(-d).clip(lower=0)
        ag=g.ewm(alpha=1/n,adjust=False).mean(); al=ls.ewm(alpha=1/n,adjust=False).mean()
        return (100-100/(1+ag/(al+1e-9)))/100
    e26=_ema(c,26); ml=(e12-e26)/c; ms=_ema(ml,9)
    tp=(h+l+c)/3; lo14=l.rolling(14).min(); hi14=h.rolling(14).max()
    lo21=l.rolling(21).min(); hi21=h.rolling(21).max()
    mfv=((c-l)-(h-c))/(h-l+1e-9)*v
    mfr=(mfv.where(tp>tp.shift(1),0).rolling(14).sum()/
         (mfv.where(tp<tp.shift(1),0).rolling(14).sum().abs()+1e-9))
    stp=_sma(tp,20); mad=tp.rolling(20).apply(lambda x:np.abs(x-x.mean()).mean(),raw=True)
    frames.append(pd.DataFrame({'RSI14':rsi(c,14),'RSI7':rsi(c,7),
        'MACD_line':ml,'MACD_signal':ms,'MACD_hist':ml-ms,
        'ROC5':c.pct_change(5),'ROC10':c.pct_change(10),'ROC20':c.pct_change(20),
        'Stoch_K14':(c-lo14)/(hi14-lo14+1e-9),
        'Stoch_D14':((c-lo14)/(hi14-lo14+1e-9)).rolling(3).mean(),
        'Stoch_K21':(c-lo21)/(hi21-lo21+1e-9),
        'Stoch_D21':((c-lo21)/(hi21-lo21+1e-9)).rolling(3).mean(),
        'WilliamsR':(hi14-c)/(hi14-lo14+1e-9),
        'CCI20':((tp-stp)/(0.015*mad+1e-9))/200,
        'MFI14':(100-100/(1+mfr))/100,'DPO10':(c.shift(6)-_sma(c,10))/c},
        index=raw.index))
    # F4 Volatility
    tr=_tr(raw); mid20=_sma(c,20); std20=c.rolling(20).std()
    atr20v=tr.ewm(span=20,adjust=False).mean()
    log_hl=np.log(h/l)**2; log_co=np.log(c/o)**2
    gk=0.5*log_hl-(2*np.log(2)-1)*log_co
    log_oc=np.log(o/c.shift(1))**2
    rs_c=log_hl*0.5-log_co*(2*np.log(2)-1); k=0.34/(1.34+11/9)
    yz=log_oc+k*log_co+(1-k)*rs_c
    r1=c.pct_change(); r2=r1**2
    v5=r1.rolling(5).std(); v20=r1.rolling(20).std()
    frames.append(pd.DataFrame({'ATR7':tr.ewm(span=7,adjust=False).mean()/c,
        'ATR14':tr.ewm(span=14,adjust=False).mean()/c,
        'ATR21':tr.ewm(span=21,adjust=False).mean()/c,
        'BB_Width20':(4*std20)/(mid20+1e-9),
        'BB_Pct20':(c-(mid20-2*std20))/(4*std20+1e-9),
        'KC_Pct':(c-(_ema(c,20)-2*atr20v))/(4*atr20v+1e-9),
        'GK_Vol10':np.sqrt(gk.rolling(10).mean()),
        'YZ_Vol10':np.sqrt(yz.rolling(10).mean()),
        'RolStd5':v5,'RolStd10':r1.rolling(10).std(),
        'RolStd20':v20,'RolStd60':r1.rolling(60).std(),
        'GARCH_proxy':0.85*r2.ewm(span=20,adjust=False).mean()+0.1*r2,
        'VolRatio_5_20':v5/(v20+1e-9)},index=raw.index))
    # F5 Volume
    obv=(np.sign(c.diff())*v).cumsum()
    tp2=(h+l+c)/3; vwap=(tp2*v).rolling(20).sum()/v.rolling(20).sum()
    mfv2=((c-l)-(h-c))/(h-l+1e-9)*v
    ad=mfv2.cumsum(); pvt=(c.pct_change()*v).cumsum(); fi=c.diff()*v
    frames.append(pd.DataFrame({'OBV_Diff':obv.diff(),
        'OBV_ZScore':(obv-obv.rolling(20).mean())/(obv.rolling(20).std()+1e-9),
        'VWAP_Dist':(c-vwap)/c,
        'CMF20':mfv2.rolling(20).sum()/(v.rolling(20).sum()+1e-9),
        'AD_ZScore':(ad-ad.rolling(20).mean())/(ad.rolling(20).std()+1e-9),
        'PVT_ZScore':(pvt-pvt.rolling(20).mean())/(pvt.rolling(20).std()+1e-9),
        'ForceIndex13':_ema(fi,13)/(c*v.rolling(20).mean()+1e-9),
        'Vol_ZScore':(v-v.rolling(20).mean())/(v.rolling(20).std()+1e-9),
        'VolRatio20':v/v.rolling(20).mean()},index=raw.index))
    # F6 Microstructure
    rng=h-l+1e-9
    bt=pd.concat([c,o],axis=1).max(axis=1); bb=pd.concat([c,o],axis=1).min(axis=1)
    frames.append(pd.DataFrame({'HL_Range':(h-l)/c,'Body_Pct':(c-o).abs()/rng,
        'Upper_Shadow':(h-bt)/rng,'Lower_Shadow':(bb-l)/rng,
        'Close_Position':(c-l)/rng,'GapOpen':(o-c.shift(1))/c.shift(1),
        'IsDoji':((c-o).abs()/rng<0.1).astype(float),
        'Spread_Proxy':(h-l)/o},index=raw.index))
    # F7 Regime
    hi252=c.rolling(252).max(); lo252=c.rolling(252).min()
    up_m=h-h.shift(1); dn_m=l.shift(1)-l
    pdm=up_m.where((up_m>dn_m)&(up_m>0),0)
    ndm=dn_m.where((dn_m>up_m)&(dn_m>0),0)
    atr14t=tr.ewm(span=14,adjust=False).mean()
    pdi=_ema(pdm,14)/(atr14t+1e-9); mdi=_ema(ndm,14)/(atr14t+1e-9)
    dx=(pdi-mdi).abs()/(pdi+mdi+1e-9)
    def hurst(s,n=20):
        def rs(x):
            x=np.array(x); m=x-x.mean(); cs=np.cumsum(m)
            r=cs.max()-cs.min(); sv=x.std()+1e-9
            return np.log(r/sv)/np.log(n) if r>0 else 0.5
        return s.rolling(n).apply(rs,raw=True)
    frames.append(pd.DataFrame({'Pct_From_52Hi':(c-hi252)/hi252,
        'Pct_From_52Lo':(c-lo252)/lo252,
        'ZScore20':(c-c.rolling(20).mean())/(c.rolling(20).std()+1e-9),
        'ZScore60':(c-c.rolling(60).mean())/(c.rolling(60).std()+1e-9),
        'Drawdown20':(c-c.rolling(20).max())/(c.rolling(20).max()+1e-9),
        'ADX14':_ema(dx,14),'PlusDI14':pdi,'MinusDI14':mdi,
        'Hurst20':hurst(c.pct_change().fillna(0),20)},index=raw.index))
    # F8 Temporal
    idx = raw.index
    if hasattr(idx, 'tz') and idx.tz is not None:
        idx = idx.tz_localize(None)
    idx = pd.DatetimeIndex(idx)
    dow=idx.dayofweek.values; mon=idx.month.values
    woy=idx.isocalendar().week.values.astype(float)
    frames.append(pd.DataFrame({'DOW_sin':np.sin(2*np.pi*dow/5),
        'DOW_cos':np.cos(2*np.pi*dow/5),'Month_sin':np.sin(2*np.pi*mon/12),
        'Month_cos':np.cos(2*np.pi*mon/12),'WOY_sin':np.sin(2*np.pi*woy/52),
        'WOY_cos':np.cos(2*np.pi*woy/52),
        'Quarter_sin':np.sin(2*np.pi*((mon-1)//3)/4),
        'Quarter_cos':np.cos(2*np.pi*((mon-1)//3)/4)},index=raw.index))
    df=pd.concat(frames,axis=1)
    # F9 Interactions
    df['RSI_x_ATR']    =df['RSI14']*df['ATR14']
    df['MACD_x_Vol']   =df['MACD_hist']*df['RolStd20']
    df['VolSurp_x_Ret']=df['Vol_ZScore']*df['LogRet1']
    df['BB_RSI']       =df['BB_Pct20']*(df['RSI14']-0.5)
    df['ADX_x_MACD']   =df['ADX14']*np.sign(df['MACD_line'])
    df['Body_x_VolZ']  =df['Body_Pct']*df['Vol_ZScore']
    # Targets
    nr=df['LogRet1'].shift(-1)
    df['LogReturn_Next']=nr
    df['Direction']=np.where(nr>direction_threshold,1,np.where(nr<-direction_threshold,-1,0))
    df['Volatility_Next']=df['LogRet1'].rolling(5).std().shift(-1)
    df.dropna(inplace=True)
    return df

TARGET_COLS=['LogReturn_Next','Direction','Volatility_Next']
_EXCL=['Open','High','Low','Volume','Close',
       'SMA5','SMA10','SMA20','SMA50','SMA200',
       'EMA5','EMA10','EMA20','EMA50','EMA200',
       'DEMA10','DEMA20','TEMA12','WMA10','WMA20','VWMA20']

def get_feature_cols(df):
    return [c for c in df.columns if c not in set(TARGET_COLS+_EXCL)]

FEATURE_GROUPS={
    'Returns'       :['Ret1','LogRet1','LogRet2','LogRet5','LogRet10','LogRet20','GapRet','IntradayRet','Cum5Ret'],
    'Trend'         :['Dist_SMA10','Dist_SMA20','Dist_SMA50','Dist_SMA200','Dist_EMA10','Dist_EMA20','Dist_EMA50','Dist_EMA200','Cross_EMA5_EMA20','Cross_EMA10_EMA50','Cross_SMA50_SMA200','Dist_VWMA20'],
    'Momentum'      :['RSI14','RSI7','MACD_line','MACD_signal','MACD_hist','ROC5','ROC10','ROC20','Stoch_K14','Stoch_D14','Stoch_K21','Stoch_D21','WilliamsR','CCI20','MFI14','DPO10'],
    'Volatility'    :['ATR7','ATR14','ATR21','BB_Width20','BB_Pct20','KC_Pct','GK_Vol10','YZ_Vol10','RolStd5','RolStd10','RolStd20','RolStd60','GARCH_proxy','VolRatio_5_20'],
    'Volume'        :['OBV_Diff','OBV_ZScore','VWAP_Dist','CMF20','AD_ZScore','PVT_ZScore','ForceIndex13','Vol_ZScore','VolRatio20'],
    'Microstructure':['HL_Range','Body_Pct','Upper_Shadow','Lower_Shadow','Close_Position','GapOpen','IsDoji','Spread_Proxy'],
    'Regime'        :['Pct_From_52Hi','Pct_From_52Lo','ZScore20','ZScore60','Drawdown20','ADX14','PlusDI14','MinusDI14','Hurst20'],
    'Temporal'      :['DOW_sin','DOW_cos','Month_sin','Month_cos','WOY_sin','WOY_cos','Quarter_sin','Quarter_cos'],
    'Interaction'   :['RSI_x_ATR','MACD_x_Vol','VolSurp_x_Ret','BB_RSI','ADX_x_MACD','Body_x_VolZ'],
}


# ═══════════════════════════════════════════════════════════════════════════════
# 5 ── PREPROCESSING & CORRELATION
# ═══════════════════════════════════════════════════════════════════════════════

def preprocess(df, feature_cols, train_ratio=0.8):
    """Returns: train_df, test_df, scaler, stat_df, dist_df, redun, bounds_df"""
    for col in feature_cols:
        if df[col].isna().any(): df[col].fillna(df[col].median(), inplace=True)
        df[col].replace([np.inf,-np.inf], df[col].median(), inplace=True)
    # Stationarity
    try:
        from statsmodels.tsa.stattools import adfuller, kpss
        rows=[]
        for col in feature_cols:
            s=df[col].dropna()
            try: _,ap,*_=adfuller(s,autolag='AIC'); ao=ap<0.05
            except: ap,ao=np.nan,False
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                try: _,kp,*_=kpss(s,regression='c',nlags='auto'); ko=kp>0.05
                except: kp,ko=np.nan,False
            rows.append({'Feature':col,'ADF_p':round(float(ap),4) if not np.isnan(float(ap)) else np.nan,
                         'KPSS_p':round(float(kp),4) if not np.isnan(float(kp)) else np.nan,
                         'Stationary':ao and ko})
        stat_df=pd.DataFrame(rows)
    except ImportError:
        stat_df=pd.DataFrame({'Feature':feature_cols,'Stationary':True})
    # Distribution
    dist_rows=[]
    for col in feature_cols:
        s=df[col].dropna()
        jb,jp=stats.jarque_bera(s)
        dist_rows.append({'Feature':col,'Mean':round(s.mean(),6),'Std':round(s.std(),6),
                          'Skew':round(s.skew(),3),'Kurt':round(s.kurtosis(),3),
                          'JB_p':round(jp,4),'Normal':jp>0.05})
    dist_df=pd.DataFrame(dist_rows)
    # Winsorise
    df_w=df.copy(); bounds={}
    for col in feature_cols:
        lo=df[col].quantile(0.01); hi=df[col].quantile(0.99)
        df_w[col]=df[col].clip(lo,hi)
        bounds[col]={'lower':lo,'upper':hi,'n_clipped':int(((df[col]<lo)|(df[col]>hi)).sum())}
    bounds_df=pd.DataFrame(bounds).T
    # Scale
    split=int(len(df_w)*train_ratio)
    tr=df_w.iloc[:split].copy(); te=df_w.iloc[split:].copy()
    scaler=RobustScaler()
    tr[feature_cols]=scaler.fit_transform(tr[feature_cols])
    te[feature_cols]=scaler.transform(te[feature_cols])
    # Redundancy
    corr=tr[feature_cols].corr().abs()
    upper=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
    pairs=[(c,r,upper.loc[r,c]) for c in upper.columns for r in upper.index
           if upper.loc[r,c]>0.92]
    pairs.sort(key=lambda x:-x[2])
    tc=tr[feature_cols].corrwith(tr['LogReturn_Next']).abs()
    to_drop=set()
    for f1,f2,r in pairs:
        if f1 not in to_drop and f2 not in to_drop:
            to_drop.add(f1 if tc.get(f1,0)<tc.get(f2,0) else f2)
    redun={'pairs':pairs,'to_drop':list(to_drop),
           'to_keep':[f for f in feature_cols if f not in to_drop]}
    return tr, te, scaler, stat_df, dist_df, redun, bounds_df


# ═══════════════════════════════════════════════════════════════════════════════
# 6 ── CHART GENERATORS
# ═══════════════════════════════════════════════════════════════════════════════

def _fig_to_bytes(fig):
    buf=io.BytesIO(); fig.savefig(buf,format='png',bbox_inches='tight',dpi=150)
    buf.seek(0); plt.close(fig); return buf

def chart_price_history(df, sym):
    fig,axes=plt.subplots(3,1,figsize=(10,7),sharex=True,
                          gridspec_kw={'height_ratios':[3,1,1]})
    c=df['Close'] if 'Close' in df.columns else df.iloc[:,3]
    v=df['Volume'] if 'Volume' in df.columns else df.iloc[:,4]
    axes[0].plot(c.index,c.values,color=P_BLUE,lw=1,label='Close')
    axes[0].fill_between(c.index,c.values,alpha=0.08,color=P_BLUE)
    axes[0].set_ylabel('Price (USD)',fontsize=9); axes[0].set_title(f'{sym} — Price & Volume',fontsize=11)
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    ret=(np.log(c/c.shift(1))*100).fillna(0)
    axes[1].fill_between(ret.index,ret.values,where=ret>0,color=P_TEAL,alpha=0.7,label='Up')
    axes[1].fill_between(ret.index,ret.values,where=ret<0,color=P_CORAL,alpha=0.7,label='Down')
    axes[1].axhline(0,color=P_GRAY,lw=0.7); axes[1].set_ylabel('Log Ret %',fontsize=9)
    axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)
    axes[2].bar(v.index,v.values,color=P_GRAY,alpha=0.5,width=1)
    axes[2].set_ylabel('Volume',fontsize=9); axes[2].grid(alpha=0.3)
    plt.tight_layout(); return _fig_to_bytes(fig)

def chart_return_dist(df_feat, sym):
    fig,axes=plt.subplots(1,3,figsize=(12,4))
    lr=df_feat['LogRet1'].dropna()*100
    axes[0].hist(lr,bins=80,color=P_BLUE,edgecolor='none',alpha=0.8,density=True)
    xs=np.linspace(lr.min(),lr.max(),200)
    axes[0].plot(xs,stats.norm.pdf(xs,lr.mean(),lr.std()),color=P_CORAL,lw=1.5,label='Normal fit')
    axes[0].set_title('Daily log-return distribution',fontsize=10)
    axes[0].set_xlabel('Log return (%)',fontsize=9); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    stats.probplot(lr,dist='norm',plot=axes[1])
    axes[1].set_title('Q-Q plot vs Normal',fontsize=10); axes[1].grid(alpha=0.3)
    vc=df_feat['Direction'].value_counts().sort_index()
    lbl={-1:'Down',0:'Neutral',1:'Up'}
    axes[2].bar([lbl[i] for i in vc.index],vc.values,
                color=[P_CORAL,P_GRAY,P_TEAL],alpha=0.85,edgecolor='none')
    for i,(idx2,vv) in enumerate(vc.items()):
        axes[2].text(i,vv+5,f'{100*vv/len(df_feat):.1f}%',ha='center',fontsize=8)
    axes[2].set_title('Direction class balance',fontsize=10); axes[2].grid(axis='y',alpha=0.3)
    plt.suptitle(f'{sym} — Return Distributions',fontsize=11)
    plt.tight_layout(); return _fig_to_bytes(fig)

def chart_feature_target_corr(train_df, feature_cols, sym):
    fig,axes=plt.subplots(1,2,figsize=(14,max(6,len(feature_cols)*0.15)))
    for ax,tgt in zip(axes,['LogReturn_Next','Direction']):
        corrs=train_df[feature_cols].corrwith(train_df[tgt]).sort_values()
        colors_bar=[P_CORAL if v<0 else P_TEAL for v in corrs.values]
        corrs.plot(kind='barh',ax=ax,color=colors_bar,edgecolor='none')
        ax.axvline(0,color=P_GRAY,lw=0.8)
        ax.axvline(0.05,color=P_AMBER,lw=0.7,ls='--',alpha=0.6)
        ax.axvline(-0.05,color=P_AMBER,lw=0.7,ls='--',alpha=0.6)
        ax.set_title(f'Feature → {tgt}',fontsize=10)
        ax.set_xlabel('Pearson r',fontsize=9); ax.tick_params(labelsize=7); ax.grid(axis='x',alpha=0.3)
    plt.suptitle(f'{sym} — Feature-Target Correlations',fontsize=11)
    plt.tight_layout(); return _fig_to_bytes(fig)

def chart_corr_heatmap(train_df, feature_cols, sym, method='pearson'):
    cols=feature_cols[:50]
    corr=train_df[cols].corr(method=method)
    n=len(cols); fs=max(10,n*0.2)
    fig,ax=plt.subplots(figsize=(fs,fs*0.85))
    mask=np.triu(np.ones_like(corr,dtype=bool))
    sns.heatmap(corr,mask=mask,cmap=sns.diverging_palette(220,20,as_cmap=True),
                center=0,vmin=-1,vmax=1,ax=ax,square=True,linewidths=0.2,
                cbar_kws={'shrink':0.5,'label':f'{method.capitalize()} r'})
    ax.set_title(f'{sym} — {method.capitalize()} Correlation Heatmap ({n} features)',fontsize=11,pad=10)
    ax.tick_params(labelsize=6); plt.tight_layout(); return _fig_to_bytes(fig)

def chart_rolling_corr(train_df, feature_cols, sym, window=252, top_n=8):
    top=train_df[feature_cols].corrwith(train_df['LogReturn_Next']).abs().nlargest(top_n).index.tolist()
    roll=pd.DataFrame({f:train_df[f].rolling(window).corr(train_df['LogReturn_Next'])
                       for f in top},index=train_df.index)
    fig,ax=plt.subplots(figsize=(12,5))
    colors_r=plt.cm.tab10(np.linspace(0,1,top_n))
    for i,col in enumerate(top):
        roll[col].plot(ax=ax,color=colors_r[i],lw=1,label=col,alpha=0.85)
    ax.axhline(0,color=P_GRAY,lw=0.8,ls='--')
    ax.set_title(f'{sym} — Rolling {window}-day Correlation: Top {top_n} Features → LogReturn_Next',fontsize=10)
    ax.set_ylabel('Pearson r',fontsize=9); ax.legend(fontsize=7,ncol=2); ax.grid(alpha=0.3)
    plt.tight_layout(); return _fig_to_bytes(fig)

def chart_family_corr(train_df, sym):
    tgts=['LogReturn_Next','Direction','Volatility_Next']
    data=[]
    for fam,feats in FEATURE_GROUPS.items():
        ex=[f for f in feats if f in train_df.columns]
        for tgt in tgts:
            data.append({'Family':fam,'Target':tgt,
                         'r':train_df[ex].corrwith(train_df[tgt]).abs().mean() if ex else 0})
    fdf=pd.DataFrame(data)
    fig,ax=plt.subplots(figsize=(12,5))
    x=np.arange(len(FEATURE_GROUPS)); w=0.25
    for i,(tgt,col) in enumerate(zip(tgts,[P_TEAL,P_AMBER,P_CORAL])):
        vals=fdf[fdf['Target']==tgt].set_index('Family')['r']
        vals=vals.reindex(list(FEATURE_GROUPS.keys()),fill_value=0)
        ax.bar(x+i*w,vals.values,w,label=tgt,color=col,alpha=0.85,edgecolor='none')
    ax.set_xticks(x+w); ax.set_xticklabels(list(FEATURE_GROUPS.keys()),rotation=25,ha='right',fontsize=9)
    ax.set_ylabel('Mean |Pearson r|',fontsize=10)
    ax.set_title(f'{sym} — Feature Family Correlation Strength vs Targets',fontsize=11)
    ax.legend(fontsize=9); ax.grid(axis='y',alpha=0.3); plt.tight_layout()
    return _fig_to_bytes(fig)

def chart_volatility_regime(df_feat, sym):
    fig,axes=plt.subplots(2,1,figsize=(12,6),sharex=True)
    rv=df_feat['RolStd20'].dropna()*100
    axes[0].fill_between(rv.index,rv.values,color=P_AMBER,alpha=0.6,label='20-day vol')
    axes[0].plot(rv.index,rv.rolling(60).mean(),color=P_CORAL,lw=1,label='60-day MA')
    axes[0].set_ylabel('Realised Vol (%)',fontsize=9); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[0].set_title(f'{sym} — Volatility Regime',fontsize=11)
    hurst=df_feat['Hurst20'].dropna()
    axes[1].plot(hurst.index,hurst.values,color=P_BLUE,lw=0.8,alpha=0.8)
    axes[1].axhline(0.5,color=P_CORAL,lw=1,ls='--',label='H=0.5 (random)')
    axes[1].fill_between(hurst.index,hurst.values,0.5,
                         where=hurst>0.5,alpha=0.2,color=P_TEAL,label='Trending (H>0.5)')
    axes[1].fill_between(hurst.index,hurst.values,0.5,
                         where=hurst<0.5,alpha=0.2,color=P_CORAL,label='Mean-rev (H<0.5)')
    axes[1].set_ylabel('Hurst Exponent',fontsize=9); axes[1].legend(fontsize=7)
    axes[1].set_ylim(0,1); axes[1].grid(alpha=0.3)
    plt.tight_layout(); return _fig_to_bytes(fig)


# ═══════════════════════════════════════════════════════════════════════════════
# 7 ── REPORTLAB PDF
# ═══════════════════════════════════════════════════════════════════════════════

def build_pdf(all_results, tickers, out_path, fetch_summary_rows):
    W, H = A4
    M = 2*cm

    doc = SimpleDocTemplate(
        out_path, pagesize=A4,
        leftMargin=M, rightMargin=M,
        topMargin=M, bottomMargin=M,
        title="Finance World Model — Research Report",
        author="Finance Research Pipeline",
    )

    base = getSampleStyleSheet()

    def S(name, **kw):
        return ParagraphStyle(name, parent=base['Normal'], **kw)

    sTitle    = S('sTitle',    fontSize=26, textColor=R_NAVY,  fontName='Helvetica-Bold',
                  alignment=TA_CENTER, spaceAfter=6)
    sSubtitle = S('sSubt',     fontSize=13, textColor=R_BLUE,  fontName='Helvetica',
                  alignment=TA_CENTER, spaceAfter=4)
    sMeta     = S('sMeta',     fontSize=9,  textColor=R_GRAY,  alignment=TA_CENTER, spaceAfter=2)
    sH1       = S('sH1',       fontSize=16, textColor=R_NAVY,  fontName='Helvetica-Bold',
                  spaceBefore=14, spaceAfter=6, borderPad=2)
    sH2       = S('sH2',       fontSize=13, textColor=R_BLUE,  fontName='Helvetica-Bold',
                  spaceBefore=10, spaceAfter=4)
    sH3       = S('sH3',       fontSize=11, textColor=R_TEAL,  fontName='Helvetica-Bold',
                  spaceBefore=6, spaceAfter=3)
    sBody     = S('sBody',     fontSize=9,  textColor=R_BLACK, leading=14, spaceAfter=4,
                  alignment=TA_JUSTIFY)
    sCaption  = S('sCaption',  fontSize=8,  textColor=R_GRAY,  alignment=TA_CENTER,
                  fontName='Helvetica-Oblique', spaceAfter=6)
    sBullet   = S('sBullet',   fontSize=9,  textColor=R_BLACK, leading=13, spaceAfter=2,
                  leftIndent=14, bulletIndent=4)
    sFootnote = S('sFootnote', fontSize=7,  textColor=R_GRAY,  alignment=TA_CENTER)

    story = []

    def hr(color=R_LGRAY, thickness=0.5, space=4):
        return HRFlowable(width='100%', thickness=thickness,
                          color=color, spaceAfter=space, spaceBefore=space)

    def img(buf, width=None, caption=None, max_height=None):
        buf.seek(0)
        avail_w = width or (W - 2*M)
        avail_h = max_height or (H - 4*M)
        im = Image(buf)
        aspect = im.imageHeight / max(im.imageWidth, 1)
        fit_w  = avail_w
        fit_h  = fit_w * aspect
        if fit_h > avail_h:
            fit_h = avail_h
            fit_w = fit_h / aspect
        im.drawWidth  = fit_w
        im.drawHeight = fit_h
        im.hAlign = 'CENTER'
        items = [im]
        if caption: items.append(Paragraph(caption, sCaption))
        return items

    def section_header(text):
        return [hr(R_NAVY, 1.5, 6), Paragraph(text, sH1), hr(R_LGRAY, 0.5, 4)]

    def tbl(data, col_widths, head_rows=1):
        t = Table(data, colWidths=col_widths, repeatRows=head_rows)
        style = [
            ('BACKGROUND',  (0,0), (-1,head_rows-1), R_NAVY),
            ('TEXTCOLOR',   (0,0), (-1,head_rows-1), R_WHITE),
            ('FONTNAME',    (0,0), (-1,head_rows-1), 'Helvetica-Bold'),
            ('FONTSIZE',    (0,0), (-1,-1), 8),
            ('ALIGN',       (0,0), (-1,-1), 'CENTER'),
            ('VALIGN',      (0,0), (-1,-1), 'MIDDLE'),
            ('ROWBACKGROUNDS', (0,head_rows), (-1,-1), [R_BG, R_WHITE]),
            ('GRID',        (0,0), (-1,-1), 0.3, R_LGRAY),
            ('TOPPADDING',  (0,0), (-1,-1), 4),
            ('BOTTOMPADDING',(0,0),(-1,-1), 4),
        ]
        t.setStyle(TableStyle(style))
        return t

    # ── COVER PAGE ────────────────────────────────────────────────────────────
    story += [Spacer(1, 3*cm)]
    story.append(Paragraph("Finance World Model", sTitle))
    story.append(Paragraph("Research-Grade Feature Analysis Report", sSubtitle))
    story += [Spacer(1, 0.5*cm), hr(R_BLUE, 2, 8), Spacer(1, 0.3*cm)]
    story.append(Paragraph(
        f"Generated: {datetime.now().strftime('%B %d, %Y at %H:%M UTC')}", sMeta))
    story.append(Paragraph(
        f"Companies analysed: {len(all_results)}   "
        f"Feature families: 9   Features per ticker: 91   "
        f"Targets: LogReturn, Direction, Volatility", sMeta))
    story += [Spacer(1, 1.5*cm)]

    grid_cols = 6
    ticker_list = list(all_results.keys())
    rows_tbl = [ticker_list[i:i+grid_cols] for i in range(0, len(ticker_list), grid_cols)]
    if rows_tbl:
        cw = (W - 2*M) / grid_cols
        grid_data = [[Paragraph(t, S('tc2', fontSize=7, alignment=TA_CENTER))
                      for t in row] + [Paragraph('', base['Normal'])]*(grid_cols-len(row))
                     for row in rows_tbl]
        story.append(tbl([[Paragraph('Ticker', S('h', fontSize=7, textColor=R_WHITE,
                      fontName='Helvetica-Bold', alignment=TA_CENTER))]*grid_cols]
                     + grid_data, [cw]*grid_cols))

    story += [Spacer(1, 1*cm), hr(R_LGRAY, 0.5, 4)]
    story.append(Paragraph(
        "This report was generated programmatically from Yahoo Finance OHLCV data. "
        "All analysis is performed on historically available data only. "
        "No forward-looking information is included. For research purposes only.", sFootnote))
    story.append(PageBreak())

    # ── SECTION 1: EXECUTIVE SUMMARY ─────────────────────────────────────────
    story += section_header("1. Executive Summary")
    story.append(Paragraph(
        "This report presents a comprehensive feature engineering and correlation analysis "
        "across multiple equity tickers. For each company, 91 quantitative features are "
        "computed from raw OHLCV data spanning up to 25 years, covering nine families: "
        "returns, trend, momentum, volatility, volume, microstructure, market regime, "
        "temporal encoding, and interaction terms. The pipeline applies stationarity "
        "testing, distribution analysis, outlier winsorisation, and RobustScaler "
        "normalisation before performing Pearson and Spearman correlation analysis "
        "against three prediction targets.", sBody))
    story.append(Spacer(1, 6))

    sum_rows = [[Paragraph(h, S('sh',fontSize=8,textColor=R_WHITE,
                  fontName='Helvetica-Bold',alignment=TA_CENTER))
                 for h in ['Ticker','Sector','Train Rows','Test Rows',
                           'Features','Pruned','Stat. (%)',
                           'High Skew','Class -1 (%)','Class 0 (%)','Class +1 (%)']]]
    cw_sum = [(W-2*M)/11]*11
    for sym, res in all_results.items():
        tr2 = res.get('train_df'); df_f = res.get('df_feat')
        stat_df = res.get('stat_df'); dist_df = res.get('dist_df')
        redun = res.get('redun',{}); fc = res.get('feature_cols',[])
        if tr2 is None or df_f is None: continue
        n_stat = int(stat_df['Stationary'].sum()) if stat_df is not None and 'Stationary' in stat_df else 0
        n_feat = len(stat_df) if stat_df is not None else len(fc)
        pct_stat = f"{100*n_stat/max(n_feat,1):.0f}" if n_feat>0 else '–'
        n_skew = int((dist_df['Skew'].abs()>2).sum()) if dist_df is not None else '–'
        te2 = res.get('test_df')
        vc = df_f['Direction'].value_counts()
        n2 = len(df_f)
        def pct(k): return f"{100*vc.get(k,0)/max(n2,1):.1f}"
        row = [sym, TICKER_SECTOR.get(sym,'')[:12],
               str(len(tr2)) if tr2 is not None else '–',
               str(len(te2)) if te2 is not None else '–',
               str(len(fc)), str(len(redun.get('to_keep',[]))),
               pct_stat, str(n_skew), pct(-1), pct(0), pct(1)]
        sum_rows.append([Paragraph(str(v), S('sv',fontSize=7,alignment=TA_CENTER)) for v in row])
    story.append(tbl(sum_rows, cw_sum))
    story += [Spacer(1,6), hr(), PageBreak()]

    # ── SECTION 2: FETCH STATUS ───────────────────────────────────────────────
    story += section_header("2. Data Fetch Status")
    story.append(Paragraph(
        "The following table summarises the yfinance data fetch results. "
        "Each row shows which dataset types were successfully retrieved per ticker.", sBody))
    story += [Spacer(1,6)]

    if fetch_summary_rows:
        ds_labels = ['HIS','INF','BAL','INC','CSH','REC','DIV','SPL']
        fh = ['Ticker','Sector','Result','Retries'] + ds_labels
        fetch_tbl_rows = [[Paragraph(h, S('fh',fontSize=7,textColor=R_WHITE,
                           fontName='Helvetica-Bold',alignment=TA_CENTER)) for h in fh]]
        ds_keys = ['historical','info','balance_sheet','income_statement',
                   'cash_flow','recommendations','dividends','splits']
        for r in fetch_summary_rows[:80]:
            res_val = str(r.get('result','–'))
            res_col = R_TEAL if res_val=='successful' else (R_CORAL if res_val=='failed' else R_AMBER)
            row_data = [
                Paragraph(str(r.get('symbol',''))[:8], S('fc2',fontSize=7,fontName='Helvetica-Bold',alignment=TA_CENTER)),
                Paragraph(TICKER_SECTOR.get(str(r.get('symbol','')),'')[:12], S('fs',fontSize=6,alignment=TA_CENTER)),
                Paragraph(res_val[:4].upper(), S('fr',fontSize=7,textColor=res_col,fontName='Helvetica-Bold',alignment=TA_CENTER)),
                Paragraph(str(r.get('retries',0)), S('ft',fontSize=7,alignment=TA_CENTER)),
            ]
            for dk in ds_keys:
                v = r.get('datasets',{}).get(dk,r.get(dk,'–'))
                icon = '✓' if v is True else ('✗' if v is False else '–')
                ic = R_TEAL if v is True else (R_CORAL if v is False else R_GRAY)
                row_data.append(Paragraph(icon, S('fi',fontSize=8,textColor=ic,alignment=TA_CENTER)))
            fetch_tbl_rows.append(row_data)
        cw_f = [1.5*cm, 2.5*cm, 1.5*cm, 1.2*cm] + [1.1*cm]*8
        story.append(tbl(fetch_tbl_rows, cw_f))
    story += [Spacer(1,6), hr(), PageBreak()]

    # ── SECTION 3: PER-TICKER ANALYSIS ───────────────────────────────────────
    story += section_header("3. Per-Ticker Analysis")

    for sym, res in all_results.items():
        df_raw  = res.get('df_raw');  df_feat = res.get('df_feat')
        train_df= res.get('train_df'); test_df = res.get('test_df')
        fc      = res.get('feature_cols',[])
        redun   = res.get('redun',{})
        stat_df = res.get('stat_df');  dist_df = res.get('dist_df')
        if df_raw is None or df_feat is None: continue

        story.append(Paragraph(f"3.{list(all_results.keys()).index(sym)+1} — {sym}", sH2))
        story.append(Paragraph(
            f"Sector: {TICKER_SECTOR.get(sym,'N/A')}   "
            f"Period: {df_raw.index[0].date()} to {df_raw.index[-1].date()}   "
            f"Trading days: {len(df_raw):,}   "
            f"Feature rows (post-dropna): {len(df_feat):,}", sBody))
        story += [Spacer(1,4)]

        story.append(Paragraph("Price History & Returns", sH3))
        story += img(chart_price_history(df_raw, sym),
                     caption=f"Figure: {sym} daily close price, log returns, and volume (full history)")

        story.append(Paragraph("Return Distribution & Target Balance", sH3))
        story += img(chart_return_dist(df_feat, sym),
                     caption=f"Figure: {sym} log-return histogram vs normal, Q-Q plot, direction class balance")

        lr = df_feat['LogRet1'].dropna()*100
        vc = df_feat['Direction'].value_counts()
        n3  = len(df_feat)
        stats_data = [
            [Paragraph(h, S('sh2',fontSize=8,textColor=R_WHITE,fontName='Helvetica-Bold',alignment=TA_CENTER))
             for h in ['Metric','Value','Metric','Value']],
            ['Annual Ret (%)', f"{lr.mean()*252:.2f}",  'Ann. Vol (%)', f"{lr.std()*np.sqrt(252):.2f}"],
            ['Sharpe (approx)', f"{lr.mean()/lr.std()*np.sqrt(252):.3f}", 'Skewness', f"{lr.skew():.3f}"],
            ['Kurtosis', f"{lr.kurtosis():.3f}", 'Max drawdown', f"{float((df_feat['Drawdown20'].min()*100)):.2f}%"],
            ['Up days (%)', f"{100*vc.get(1,0)/n3:.1f}", 'Neutral (%)', f"{100*vc.get(0,0)/n3:.1f}"],
            ['Down days (%)', f"{100*vc.get(-1,0)/n3:.1f}", 'Hurst (mean)', f"{df_feat['Hurst20'].mean():.3f}"],
        ]
        cw2 = [(W-2*M)/4]*4
        for i in range(1, len(stats_data)):
            stats_data[i] = [Paragraph(str(v), S('sv2',fontSize=8,alignment=TA_CENTER))
                             for v in stats_data[i]]
        story.append(tbl(stats_data, cw2))
        story += [Spacer(1,6)]

        story.append(Paragraph("Feature Engineering Summary", sH3))
        fe_data = [[Paragraph(h, S('feh',fontSize=8,textColor=R_WHITE,fontName='Helvetica-Bold',
                    alignment=TA_CENTER)) for h in ['Family','Features','Stationary','High Skew','Notes']]]
        for fam, feats in FEATURE_GROUPS.items():
            ex = [f for f in feats if f in fc]
            n_stat_fam = 0
            if stat_df is not None and 'Stationary' in stat_df:
                n_stat_fam = stat_df[stat_df['Feature'].isin(ex)]['Stationary'].sum()
            n_skew_fam = 0
            if dist_df is not None:
                n_skew_fam = (dist_df[dist_df['Feature'].isin(ex)]['Skew'].abs()>2).sum()
            notes = {'Returns':'Stationary by construction','Trend':'Scale-free (dist/price)',
                     'Momentum':'Bounded oscillators','Volatility':'log1p transforms applied',
                     'Volume':'Z-scored for stationarity','Microstructure':'Candle geometry, 0-1 bounded',
                     'Regime':'Long-window regime indicators','Temporal':'Sin/cos cyclic encoding',
                     'Interaction':'Cross-family non-linear terms'}.get(fam,'')
            fe_data.append([Paragraph(v, S('fev',fontSize=8,alignment=TA_CENTER))
                            for v in [fam, str(len(ex)), str(int(n_stat_fam)),
                                      str(int(n_skew_fam)), notes]])
        cw_fe = [2.5*cm, 1.8*cm, 2*cm, 2*cm, (W-2*M-8.3*cm)]
        story.append(tbl(fe_data, cw_fe))
        story += [Spacer(1,6)]

        story.append(Paragraph("Correlation Analysis", sH3))
        story += img(chart_corr_heatmap(train_df, fc, sym, 'pearson'),
                     caption=f"Figure: {sym} Pearson correlation heatmap (lower triangle, train set, top 50 features)")
        story += img(chart_corr_heatmap(train_df, fc, sym, 'spearman'),
                     caption=f"Figure: {sym} Spearman correlation heatmap (non-linear rank correlations)")

        to_drop = redun.get('to_drop',[])
        pairs   = redun.get('pairs',[])[:10]
        if pairs:
            story.append(Paragraph(
                f"<b>Redundant features (|r| > 0.92):</b> {len(to_drop)} features dropped from "
                f"{len(redun.get('pairs',[]))} correlated pairs. Top pairs:", sBody))
            for f1, f2, r in pairs[:6]:
                story.append(Paragraph(f"• {f1} ↔ {f2}  r={r:.3f}", sBullet))
        story += [Spacer(1,4)]

        story.append(Paragraph("Feature-Target Correlations", sH3))
        story += img(chart_feature_target_corr(train_df, fc, sym),
                     caption=f"Figure: {sym} — Pearson r of each feature vs LogReturn_Next and Direction "
                             f"(dashed lines at ±0.05, teal=positive, coral=negative)")
        story += img(chart_family_corr(train_df, sym),
                     caption=f"Figure: {sym} — Mean absolute correlation per feature family vs each target")

        story.append(Paragraph("Rolling Correlation Stability", sH3))
        story += img(chart_rolling_corr(train_df, fc, sym),
                     caption=f"Figure: {sym} — 252-day rolling Pearson r, top 8 features vs LogReturn_Next. "
                             f"Stable lines = consistent signal; volatile lines = regime-dependent")

        story.append(Paragraph("Volatility & Market Regime", sH3))
        story += img(chart_volatility_regime(df_feat, sym),
                     caption=f"Figure: {sym} — Realised volatility (20-day rolling std) and Hurst exponent. "
                             f"H > 0.5 = trending regime; H < 0.5 = mean-reverting regime")

        story += [Spacer(1,8), hr(R_LGRAY), PageBreak()]

    # ── SECTION 4: CROSS-TICKER COMPARISON ───────────────────────────────────
    if len(all_results) > 1:
        story += section_header("4. Cross-Ticker Comparison")
        story.append(Paragraph(
            "The following charts compare key statistics across all analysed tickers.", sBody))

        syms_c=[]; ann_ret=[]; ann_vol=[]; sharpes=[]
        for sym, res in all_results.items():
            df_f=res.get('df_feat')
            if df_f is None: continue
            lr2=(df_f['LogRet1'].dropna()*100)
            ar=lr2.mean()*252; av=lr2.std()*np.sqrt(252)
            syms_c.append(sym); ann_ret.append(ar); ann_vol.append(av)
            sharpes.append(ar/av if av>0 else 0)

        if syms_c:
            fig,axes=plt.subplots(1,2,figsize=(14,5))
            sc=axes[0].scatter(ann_vol,ann_ret,c=sharpes,cmap='RdYlGn',
                               s=80,alpha=0.85,edgecolors=P_GRAY,lw=0.5)
            for i,(s,x,y) in enumerate(zip(syms_c,ann_vol,ann_ret)):
                axes[0].annotate(s,(x,y),textcoords='offset points',xytext=(4,3),fontsize=7)
            axes[0].axhline(0,color=P_GRAY,lw=0.7,ls='--')
            axes[0].set_xlabel('Annualised Vol (%)',fontsize=10)
            axes[0].set_ylabel('Annualised Return (%)',fontsize=10)
            axes[0].set_title('Risk-Return Profile',fontsize=11)
            plt.colorbar(sc,ax=axes[0],label='Sharpe ratio')
            axes[0].grid(alpha=0.3)
            idx_sort=np.argsort(sharpes)[::-1]
            colors_b=[P_TEAL if s>0 else P_CORAL for s in [sharpes[i] for i in idx_sort]]
            axes[1].barh([syms_c[i] for i in idx_sort],[sharpes[i] for i in idx_sort],
                         color=colors_b,edgecolor='none',alpha=0.85)
            axes[1].axvline(0,color=P_GRAY,lw=0.8)
            axes[1].set_xlabel('Sharpe Ratio (approx)',fontsize=10)
            axes[1].set_title('Sharpe Ratio Ranking',fontsize=11)
            axes[1].tick_params(labelsize=8); axes[1].grid(axis='x',alpha=0.3)
            plt.tight_layout()
            story += img(_fig_to_bytes(fig),
                         caption="Figure: Risk-return scatter (colour = Sharpe) and Sharpe ratio ranking")

            fig2,ax2=plt.subplots(figsize=(max(8,len(syms_c)*0.5),4))
            hurst_means=[all_results[s]['df_feat']['Hurst20'].mean()
                         for s in syms_c if all_results[s].get('df_feat') is not None]
            colors_h=[P_TEAL if h>0.5 else P_CORAL for h in hurst_means]
            ax2.bar(syms_c,hurst_means,color=colors_h,alpha=0.85,edgecolor='none')
            ax2.axhline(0.5,color=P_GRAY,lw=1,ls='--',label='H=0.5 (random walk)')
            ax2.set_ylabel('Mean Hurst Exponent',fontsize=10)
            ax2.set_title('Market Regime: Mean Hurst Exponent by Ticker',fontsize=11)
            ax2.set_ylim(0.3,0.8); ax2.tick_params(axis='x',rotation=45,labelsize=8)
            ax2.legend(fontsize=9); ax2.grid(axis='y',alpha=0.3); plt.tight_layout()
            story += img(_fig_to_bytes(fig2),
                         caption="Figure: Mean Hurst exponent per ticker. Green (H>0.5) = trending; Red (H<0.5) = mean-reverting")

        story += [hr(), PageBreak()]

    # ── SECTION 5: METHODOLOGY ────────────────────────────────────────────────
    story += section_header("5. Methodology")
    methodology_text = [
        ("Data Source", "Yahoo Finance via yfinance library. Adjusted OHLCV data, 25-year history, daily frequency. "
         "Parallel fetch with exponential-backoff retry and resume support."),
        ("Feature Engineering", "91 features across 9 families computed per ticker. All trend/MA features expressed "
         "as (Close - MA) / Close to ensure scale-free stationarity. ATR and volatility features normalised by price. "
         "Volume features z-scored. Temporal features encoded cyclically with sin/cos to preserve calendar distance."),
        ("Stationarity", "Augmented Dickey-Fuller (ADF) and KPSS tests applied to each feature. A feature is "
         "classified as stationary only if it passes both tests (ADF p < 0.05 and KPSS p > 0.05)."),
        ("Outlier Treatment", "Winsorisation at 1st and 99th percentiles applied per feature, computed on the full "
         "dataset before train/test split. Clip bounds stored for inference-time application."),
        ("Scaling", "RobustScaler (median/IQR) fitted on training set only and applied to test set. "
         "RobustScaler chosen over MinMaxScaler for its resistance to remaining outliers post-winsorisation."),
        ("Redundancy Pruning", "Pearson correlation computed on training set. For each pair with |r| > 0.92, "
         "the feature with lower absolute correlation to LogReturn_Next is dropped. Greedy sequential elimination."),
        ("Correlation Analysis", "Pearson (linear) and Spearman (rank-based, non-linear) correlations computed. "
         "Rolling 252-day window used to assess temporal stability. All analysis performed on training split only."),
        ("Target Variables", "LogReturn_Next = log(Close[t+1]/Close[t]). Direction = sign(LogReturn_Next) with "
         "±0.5% threshold (3-class). Volatility_Next = rolling 5-day std of log returns, shifted -1."),
    ]
    for title, body in methodology_text:
        story.append(Paragraph(f"<b>{title}:</b> {body}", sBody))
        story += [Spacer(1,4)]

    story += [hr(), PageBreak()]

    # ── SECTION 6: FEATURE REFERENCE ─────────────────────────────────────────
    story += section_header("6. Feature Reference Table")
    story.append(Paragraph(
        "Complete list of engineered features grouped by family, with definitions.", sBody))
    story += [Spacer(1,6)]

    feat_ref = {
        'Returns':        [('LogRet1','log(Close/Close[t-1])'),('LogRet2-20','Multi-lag log returns'),
                           ('GapRet','log(Open/Close[t-1]) — overnight gap'),
                           ('IntradayRet','log(Close/Open) — intraday move'),('Cum5Ret','5-day cumulative pct change')],
        'Trend':          [('Dist_SMAn','(Close - SMAn) / Close — distance from n-day SMA'),
                           ('Dist_EMAn','(Close - EMAn) / Close — distance from n-day EMA'),
                           ('Cross_EMA5_20','(EMA5 - EMA20)/Close — short/mid crossover'),
                           ('Cross_SMA50_200','(SMA50 - SMA200)/Close — golden/death cross'),
                           ('Dist_VWMA20','Distance from volume-weighted MA')],
        'Momentum':       [('RSI14/7','Relative Strength Index (Wilder), scaled 0-1'),
                           ('MACD_line','(EMA12 - EMA26)/Close — normalised MACD'),
                           ('MACD_hist','MACD line minus signal line'),
                           ('Stoch_K14/21','Stochastic oscillator %K'),
                           ('WilliamsR','Williams %R — overbought/oversold'),
                           ('CCI20','Commodity Channel Index / 200'),('MFI14','Money Flow Index 0-1')],
        'Volatility':     [('ATR7/14/21','Average True Range / Close'),
                           ('BB_Width20','Bollinger Band width / mid'),
                           ('BB_Pct20','%B position within Bollinger Bands'),
                           ('GK_Vol10','Garman-Klass volatility (OHLC-efficient)'),
                           ('YZ_Vol10','Yang-Zhang volatility estimator'),
                           ('RolStd5-60','Rolling std of returns (multiple windows)'),
                           ('GARCH_proxy','GARCH(1,1) proxy using EWMA')],
        'Volume':         [('OBV_ZScore','On-Balance Volume z-scored vs 20-day'),
                           ('VWAP_Dist','(Close - VWAP) / Close'),
                           ('CMF20','Chaikin Money Flow 20-day'),
                           ('AD_ZScore','Accumulation/Distribution line z-score'),
                           ('Vol_ZScore','Volume z-score vs 20-day mean')],
        'Microstructure': [('HL_Range','(High - Low) / Close — intraday range'),
                           ('Body_Pct','Candle body / total range'),
                           ('Upper/Lower_Shadow','Shadow fraction of range'),
                           ('Close_Position','(Close - Low) / (High - Low) — close within range'),
                           ('IsDoji','1 if body < 10% of range')],
        'Regime':         [('Pct_From_52Hi/Lo','Distance from 52-week high/low'),
                           ('ZScore20/60','Price z-score over 20/60-day window'),
                           ('Drawdown20','Drawdown from 20-day rolling peak'),
                           ('ADX14','Average Directional Index — trend strength 0-1'),
                           ('Hurst20','Hurst exponent proxy via R/S analysis')],
        'Temporal':       [('DOW_sin/cos','Day-of-week cyclical encoding (period=5)'),
                           ('Month_sin/cos','Month cyclical encoding (period=12)'),
                           ('WOY_sin/cos','Week-of-year cyclical encoding (period=52)'),
                           ('Quarter_sin/cos','Quarter cyclical encoding (period=4)')],
        'Interaction':    [('RSI_x_ATR','RSI14 * ATR14 — momentum x volatility'),
                           ('MACD_x_Vol','MACD_hist * RolStd20 — trend x volatility'),
                           ('VolSurp_x_Ret','Vol_ZScore * LogRet1 — volume surprise x return'),
                           ('BB_RSI','BB_Pct20 * (RSI14 - 0.5) — squeeze x momentum'),
                           ('ADX_x_MACD','ADX14 * sign(MACD) — trend strength x direction')],
    }

    ref_rows = [[Paragraph(h, S('rh',fontSize=8,textColor=R_WHITE,fontName='Helvetica-Bold',
                 alignment=TA_CENTER)) for h in ['Family','Feature','Description']]]
    for fam, entries in feat_ref.items():
        for i, (feat, desc) in enumerate(entries):
            ref_rows.append([
                Paragraph(fam if i==0 else '', S('rf',fontSize=8,alignment=TA_CENTER,
                          fontName='Helvetica-Bold' if i==0 else 'Helvetica')),
                Paragraph(feat, S('rf2',fontSize=7,fontName='Helvetica-Oblique',alignment=TA_LEFT)),
                Paragraph(desc, S('rf3',fontSize=7,alignment=TA_LEFT))
            ])
    cw_ref = [2.2*cm, 3.5*cm, W-2*M-5.7*cm]
    story.append(tbl(ref_rows, cw_ref))
    story += [Spacer(1,8), hr()]

    story.append(Spacer(1, 0.5*cm))
    story.append(Paragraph(
        "End of Report — Finance World Model Research Pipeline | "
        f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M')} | "
        "Data: Yahoo Finance | Features: 91 per ticker | "
        "For research and educational purposes only.", sFootnote))

    doc.build(story)
    print(f"\n  [PDF] Report saved → {out_path}")
    return out_path


# ═══════════════════════════════════════════════════════════════════════════════
# MASTER RUNNER
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("\n" + "═"*65)
    print("  FINANCE WORLD MODEL RESEARCH PIPELINE")
    print("═"*65)

    tickers = get_user_selection()
    if not tickers:
        print("No tickers selected. Exiting.")
        sys.exit(0)

    out_dir = safe_input(f"\n  Output directory [{os.path.abspath('./finance_output')}]: ", "").strip()
    if not out_dir: out_dir = './finance_output'
    os.makedirs(out_dir, exist_ok=True)

    workers_str = safe_input("  Parallel fetch workers [6]: ", "6").strip()
    try: workers = max(1, min(int(workers_str), 20))
    except: workers = 6

    skip_str = safe_input("  Resume mode — skip existing files? [Y/n]: ", "Y").strip().lower()
    skip = skip_str not in ('n','no')

    fetch_summary_path = os.path.join(out_dir, 'fetch_summary.csv')
    retry_only = False
    if os.path.exists(fetch_summary_path):
        prev = pd.read_csv(fetch_summary_path)
        prev_failed = prev[prev['result'].isin(['failed','partial'])]['symbol'].tolist()
        if prev_failed:
            print(f"\n  Found previous run: {len(prev_failed)} failed/partial tickers")
            retry_ans = safe_input(f"  Retry only those {len(prev_failed)} tickers? [y/N]: ", "N").strip().lower()
            if retry_ans in ('y','yes'):
                tickers = [t for t in prev_failed if t in tickers]
                retry_only = True
                print(f"  Retrying: {tickers}")

    _default_pdf_dir = os.path.abspath("./finance_reports")
    pdf_dir = safe_input(f"  PDF output directory [{_default_pdf_dir}]: ", "./finance_reports").strip()
    if not pdf_dir: pdf_dir = "./finance_reports"
    os.makedirs(pdf_dir, exist_ok=True)
    pdf_name = safe_input(f"  PDF filename [finance_research_report.pdf]: ", "").strip()
    if not pdf_name: pdf_name = "finance_research_report.pdf"
    if not pdf_name.endswith('.pdf'): pdf_name += '.pdf'
    pdf_path = os.path.join(pdf_dir, pdf_name)

    print(f"\n  Config:")
    print(f"    Tickers   : {len(tickers)}")
    print(f"    Data dir  : {out_dir}")
    print(f"    PDF dir   : {pdf_dir}")
    print(f"    Workers   : {workers}")
    print(f"    Resume    : {skip}")
    print(f"    PDF file  : {pdf_path}")
    confirm = safe_input("\n  Proceed? [Y/n]: ", "Y").strip().lower()
    if confirm in ('n','no'): print("Aborted."); sys.exit(0)

    # Stage 1: Fetch
    fetch_results, tracker = fetch_all(
        tickers, out_dir, max_workers=workers,
        delay=0.15, retries=3, skip=skip
    )

    fetch_summary_rows = []
    for r in fetch_results:
        row = {'symbol':r.get('symbol',''),'result':r.get('result','failed'),
               'retries':r.get('retries',0),'datasets':r.get('datasets',{})}
        for ds in DATASETS: row[ds] = r.get('datasets',{}).get(ds,'–')
        fetch_summary_rows.append(row)

    # Stages 2-5: Feature engineering + preprocessing
    all_results = {}
    successful_syms = [r['symbol'] for r in fetch_results
                       if r.get('result') in ('successful','partial')
                       and r.get('datasets',{}).get('historical') in (True, 'skip')]
    skipped_fetch = [r['symbol'] for r in fetch_results if r['symbol'] not in successful_syms]
    if skipped_fetch:
        print(f"\n  Skipped (no historical data): {', '.join(skipped_fetch)}")

    print(f"\n{'═'*60}")
    print(f"  STAGES 2-5 — Processing {len(successful_syms)} tickers")
    print(f"{'═'*60}")

    for i, sym in enumerate(successful_syms, 1):
        path = f"{out_dir}/{sym}_historical.csv"
        if not os.path.exists(path):
            print(f"  [{i}/{len(successful_syms)}] {sym} — no CSV, skipping")
            continue
        try:
            print(f"  [{i}/{len(successful_syms)}] {sym} — engineering features ...", end='')
            raw = pd.read_csv(path, index_col=0, parse_dates=True)
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)
            raw.index = _strip_tz(raw.index)
            required = {'Open','High','Low','Close','Volume'}
            if not required.issubset(set(raw.columns)):
                print(f" SKIP (missing columns)"); continue
            df_feat = engineer_features(raw)
            fc = get_feature_cols(df_feat)
            tr, te, scaler, stat_df, dist_df, redun, bounds_df = preprocess(df_feat, fc)
            all_results[sym] = {
                'df_raw': raw, 'df_feat': df_feat,
                'train_df': tr, 'test_df': te, 'scaler': scaler,
                'feature_cols': fc, 'stat_df': stat_df,
                'dist_df': dist_df, 'redun': redun, 'bounds_df': bounds_df,
            }
            n_pruned = len(redun.get('to_keep',[]))
            print(f" OK  rows={len(df_feat):,}  features={len(fc)}->{n_pruned}")
        except Exception as e:
            print(f" FAILED: {e}")

    if not all_results:
        print("\n  No tickers processed successfully. Cannot generate PDF.")
        sys.exit(1)

    # Stage 6: Generate PDF
    print(f"\n{'═'*60}")
    print(f"  STAGE 6 — Generating ReportLab PDF ({len(all_results)} tickers)")
    print(f"{'═'*60}")
    build_pdf(all_results, tickers, pdf_path, fetch_summary_rows)

    print(f"\n{'═'*65}")
    print(f"  PIPELINE COMPLETE")
    print(f"  Tickers fetched   : {len(tickers)}")
    print(f"  Tickers analysed  : {len(all_results)}")
    print(f"  PDF report        : {pdf_path}")
    print(f"  CSV data          : {out_dir}/")
    print(f"{'═'*65}\n")

    return all_results, pdf_path


if __name__ == "__main__":
    main()
else:
    # When run directly in a Colab cell (not as __main__), call main() explicitly
    main()


═════════════════════════════════════════════════════════════════
  FINANCE WORLD MODEL RESEARCH PIPELINE
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
  FINANCE RESEARCH PIPELINE — Company Selection
═════════════════════════════════════════════════════════════════

  Total available tickers : 325
  Sectors                 : 12

   1. Technology                    (50 tickers)
   2. Financials                    (40 tickers)
   3. Healthcare                    (30 tickers)
   4. Consumer_Discretionary        (30 tickers)
   5. Consumer_Staples              (20 tickers)
   6. Industrials                   (30 tickers)
   7. Energy                        (30 tickers)
   8. Materials                     (20 tickers)
   9. Real_Estate                   (20 tickers)
  10. Communication_Services        (20 tickers)
  11. Utilities                     (20 tickers)
  12. ETFs_Indices                  (20 ti

In [ ]:
#!/usr/bin/env python3
"""
GPU‑enhanced benchmark on the finance feature panel (standalone).

This script is standalone for Colab: it DOES NOT import other project modules.

Expected directory layout under FINANCE_BASE_DIR (default: /content in Colab):
  - finance_output/{TICKER}_historical.csv
  - correlation_analysis/returns_matrix.csv
  - correlation_analysis/phase5_regime_labels.csv
  - correlation_analysis/correlation_matrix_full.csv (optional)
  - correlation_analysis/phase3_cluster_membership.csv (optional)

Colab deps:
  !pip install numpy pandas scikit-learn scipy torch matplotlib
"""

import os
from datetime import datetime

import numpy as np
import pandas as pd
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Missing dependency 'torch'. In Colab run: pip install torch. "
        "Locally, install PyTorch for your platform from https://pytorch.org/get-started/locally/."
    ) from e
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr

TARGET_COLS = ["LogReturn_Next", "Direction", "Volatility_Next"]


def engineer_features(raw: pd.DataFrame, direction_threshold: float = 0.005) -> pd.DataFrame:
    """
    Lightweight, Colab-friendly feature set.

    Uses only OHLCV and produces:
    - features: lagged returns + rolling vol + simple momentum
    - targets:
        - LogReturn_Next: next-day log return
        - Volatility_Next: next 5-day std of log returns (forward)
        - Direction: {-1,0,1} based on next-day return threshold
    """
    df = raw.copy()
    if "Close" not in df.columns:
        return pd.DataFrame()
    df = df.sort_index()
    c = df["Close"].astype(float)

    logret = np.log(c / c.shift(1))
    out = pd.DataFrame(index=df.index)

    # Features (no lookahead)
    out["Ret1"] = logret.shift(1)
    out["Ret2"] = logret.shift(2)
    out["Ret5"] = logret.shift(5)
    out["Mom10"] = (c / c.shift(10) - 1.0)
    out["Mom20"] = (c / c.shift(20) - 1.0)
    out["Vol5"] = logret.rolling(5).std().shift(1)
    out["Vol20"] = logret.rolling(20).std().shift(1)
    out["Range1"] = ((df["High"].astype(float) - df["Low"].astype(float)) / c).shift(1) if "High" in df.columns and "Low" in df.columns else np.nan
    out["VolChg"] = (out["Vol5"] / (out["Vol20"] + 1e-12) - 1.0)

    # Targets (forward-only)
    out["LogReturn_Next"] = logret.shift(-1)
    out["Volatility_Next"] = logret.shift(-1).rolling(5).std()
    nxt = out["LogReturn_Next"]
    out["Direction"] = np.where(
        nxt > direction_threshold,
        1,
        np.where(nxt < -direction_threshold, -1, 0),
    )

    # Drop rows where targets are missing
    out = out.dropna(subset=["LogReturn_Next", "Volatility_Next"])
    return out


def get_feature_cols(df: pd.DataFrame) -> list:
    """Infer feature columns from an engineered per-ticker frame."""
    ignore = set(TARGET_COLS)
    return [c for c in df.columns if c not in ignore]


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Colab-friendly base directory selection
BASE_DIR = os.path.abspath(
    os.environ.get(
        "FINANCE_BASE_DIR",
        "/content" if os.path.isdir("/content") else os.getcwd(),
    )
)
DATA_DIR = os.path.join(BASE_DIR, "finance_output")
CORR_DIR = os.path.join(BASE_DIR, "correlation_analysis")
OUT_DIR = os.path.join(BASE_DIR, "comparison_report")
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_END = "2024-06-30"
VAL_END = "2025-03-31"
TEST_END = "2026-03-31"

RESULTS_DIR = os.path.join(OUT_DIR, "gpu_multi_model_benchmark")
os.makedirs(RESULTS_DIR, exist_ok=True)


def regression_metrics(y_true, y_pred):
    if y_true is None or y_pred is None:
        return {}
    yt = np.asarray(y_true).ravel().astype(float)
    yp = np.asarray(y_pred).ravel().astype(float)
    return {
        "mse": float(mean_squared_error(yt, yp)),
        "mae": float(mean_absolute_error(yt, yp)),
    }


def classification_metrics(y_true, y_pred):
    if y_true is None or y_pred is None:
        return {}
    yt = np.asarray(y_true).ravel().astype(int)
    yp = np.asarray(y_pred).ravel().astype(int)
    try:
        auc = roc_auc_score(yt, yp) if len(np.unique(yt)) == 2 else 0.5
    except Exception:
        auc = 0.5
    return {
        "accuracy": float(accuracy_score(yt, yp)),
        "precision": float(precision_score(yt, yp, average="macro", zero_division=0)),
        "recall": float(recall_score(yt, yp, average="macro", zero_division=0)),
        "auc": float(auc),
    }


# ────────────────────────────────────────────────────────────────────────────────
# Data loading + panel building (standalone)
# ────────────────────────────────────────────────────────────────────────────────


def load_returns_and_meta():
    """Load returns matrix, regime labels, correlation matrix, cluster/sector."""
    ret_path = os.path.join(CORR_DIR, "returns_matrix.csv")
    if not os.path.isfile(ret_path):
        # Auto-build from finance_output OHLCV files
        os.makedirs(CORR_DIR, exist_ok=True)
        print(f"  returns_matrix.csv not found; building from {DATA_DIR} ...")
        closes = {}
        for fn in os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else []:
            if not fn.endswith("_historical.csv"):
                continue
            ticker = fn.replace("_historical.csv", "")
            try:
                # Avoid pandas "could not infer format" warnings by parsing explicitly
                df = pd.read_csv(os.path.join(DATA_DIR, fn), index_col=0)
                df.index = pd.to_datetime(df.index, errors="coerce", utc=True).tz_convert(None)
                df = df[~df.index.isna()].sort_index()
                if "Close" not in df.columns or df["Close"].dropna().empty:
                    continue
                closes[ticker] = df["Close"].astype(float)
            except Exception:
                continue
        if not closes:
            raise FileNotFoundError(
                f"Missing: {ret_path} and could not build from {DATA_DIR} (no *_historical.csv with Close)."
            )
        close_df = pd.DataFrame(closes).sort_index()
        ret = np.log(close_df / close_df.shift(1))
        ret.to_csv(ret_path)
        print(f"  Saved auto-built returns matrix -> {ret_path} (tickers={ret.shape[1]}, dates={ret.shape[0]})")

    ret = pd.read_csv(ret_path, index_col=0)
    ret.index = pd.to_datetime(ret.index, errors="coerce", utc=True).tz_convert(None)
    ret = ret[~ret.index.isna()].sort_index()
    tickers = list(ret.columns)
    dates = ret.index

    regime_path = os.path.join(CORR_DIR, "phase5_regime_labels.csv")
    if not os.path.isfile(regime_path):
        # If regime labels are missing, default everything to "medium" (1)
        print(f"  Regime labels not found; defaulting all dates to 'medium' and saving -> {regime_path}")
        pd.Series(["medium"] * len(dates), index=dates, name="rolling_mean_r").to_csv(regime_path)
    regime_raw = pd.read_csv(regime_path, index_col=0, parse_dates=True).squeeze()
    if hasattr(regime_raw, "columns") and "rolling_mean_r" in regime_raw.columns:
        regime_raw = regime_raw["rolling_mean_r"]
    regime_raw = regime_raw.reindex(dates).ffill().bfill()
    regime_map = {"low": 0, "medium": 1, "high": 2}
    regime_int = regime_raw.map(regime_map).fillna(1).astype(int)

    corr_path = os.path.join(CORR_DIR, "correlation_matrix_full.csv")
    if os.path.isfile(corr_path):
        C = pd.read_csv(corr_path, index_col=0)
    else:
        C = ret.corr()
    C = C.reindex(index=tickers, columns=tickers).fillna(0)

    cluster_path = os.path.join(CORR_DIR, "phase3_cluster_membership.csv")
    cluster, sector = {}, {}
    if os.path.isfile(cluster_path):
        cm = pd.read_csv(cluster_path)
        for _, row in cm.iterrows():
            t = row.get("ticker")
            if t in tickers:
                cluster[t] = int(row.get("empirical_cluster", 0))
                sector[t] = str(row.get("GICS_sector", "Other"))
    for t in tickers:
        if t not in sector:
            sector[t] = "Other"
        if t not in cluster:
            cluster[t] = 0

    return ret, regime_int, C, cluster, sector, tickers, dates


def build_feature_panel(tickers, dates) -> pd.DataFrame:
    """Build (date, ticker) panel of engineered features + targets."""
    rows = []
    skipped = 0
    dates = pd.DatetimeIndex(pd.to_datetime(pd.Index(dates), errors="coerce")).dropna()
    dates = dates.sort_values()
    date_min, date_max = dates.min(), dates.max()
    dates_set = set(dates.to_pydatetime())
    for i, sym in enumerate(tickers):
        if (i + 1) % 50 == 0:
            print(f"  Building features {i+1}/{len(tickers)} ...")
        path = os.path.join(DATA_DIR, f"{sym}_historical.csv")
        if not os.path.isfile(path):
            skipped += 1
            continue
        try:
            raw = pd.read_csv(path, index_col=0)
            raw.index = pd.to_datetime(raw.index, errors="coerce", utc=True).tz_convert(None)
            raw = raw[~raw.index.isna()].sort_index()
            if "Close" not in raw.columns or len(raw) < 300:
                skipped += 1
                continue
            df_sym = engineer_features(raw)
            fc = get_feature_cols(df_sym)
            if not fc or "LogReturn_Next" not in df_sym.columns:
                skipped += 1
                continue
            df_sym = df_sym.loc[
                (df_sym.index >= date_min - pd.Timedelta(days=400))
                & (df_sym.index <= date_max + pd.Timedelta(days=5))
            ]
            # Only iterate over dates that exist in both series
            for ts in df_sym.index:
                if ts.to_pydatetime() not in dates_set:
                    continue
                row = {"date": ts, "ticker": sym}
                for c in fc:
                    if c in df_sym.columns:
                        row[c] = df_sym.loc[ts, c]
                for tgt in ["LogReturn_Next", "Direction", "Volatility_Next"]:
                    if tgt in df_sym.columns:
                        row[tgt] = df_sym.loc[ts, tgt]
                if len(row) > 5:
                    rows.append(row)
        except Exception:
            skipped += 1
    if skipped:
        print(f"  [build_panel] skipped {skipped} tickers")
    if not rows:
        raise RuntimeError("No feature rows built. Check your finance_output CSVs.")
    return pd.DataFrame(rows)


def zscore_cross_sectional(panel: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """Z-score each feature cross-sectionally (across tickers) per day."""
    out = panel.copy()
    for _, grp in out.groupby("date"):
        for c in feature_cols:
            if c not in grp.columns:
                continue
            s = grp[c].astype(float)
            mu = s.mean()
            std = s.std()
            if std and std > 1e-12:
                out.loc[grp.index, c] = (s - mu) / std
            else:
                out.loc[grp.index, c] = 0.0
    return out


def encode_sector(sector_map: dict, tickers: list) -> dict:
    """Deterministic integer encoding of sector labels using LabelEncoder."""
    le = LabelEncoder()
    labels = [sector_map.get(t, "Other") for t in tickers]
    le.fit(labels)
    return {t: int(le.transform([sector_map.get(t, "Other")])[0]) for t in tickers}


def make_splits(panel: pd.DataFrame, feature_cols: list, regime_ser: pd.Series, sector_enc: dict):
    """Split panel into train/val/test by date. Attach regime + sector."""
    p = panel.set_index(["date", "ticker"])
    all_dates = sorted(p.index.get_level_values(0).unique())

    train_dates = [d for d in all_dates if d <= pd.Timestamp(TRAIN_END)]
    val_dates = [d for d in all_dates if pd.Timestamp(TRAIN_END) < d <= pd.Timestamp(VAL_END)]
    test_dates = [d for d in all_dates if pd.Timestamp(VAL_END) < d <= pd.Timestamp(TEST_END)]

    def get_split(date_list):
        mask = p.index.get_level_values(0).isin(date_list)
        P = p.loc[mask].copy().dropna(subset=feature_cols, how="all")
        X = P[feature_cols].fillna(0).values.astype(np.float32)
        sec = np.array([sector_enc.get(t, 0) for t in P.index.get_level_values(1)], dtype=np.float32).reshape(-1, 1)
        reg = regime_ser.reindex(P.index.get_level_values(0)).values.reshape(-1, 1).astype(np.float32)
        X = np.hstack([X, reg, sec])

        y_ret = P["LogReturn_Next"].values if "LogReturn_Next" in P.columns else None
        y_dir = P["Direction"].values if "Direction" in P.columns else None
        y_vol = P["Volatility_Next"].values if "Volatility_Next" in P.columns else None
        return X, y_ret, y_dir, y_vol, P.index

    return get_split(train_dates), get_split(val_dates), get_split(test_dates)


def daily_rank_ic(dates_arr, y_true, y_pred):
    """Mean daily cross-sectional Spearman rank IC and ICIR."""
    if y_true is None or y_pred is None:
        return np.nan, np.nan, pd.Series(dtype=float)
    dates = np.asarray(dates_arr)
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    daily_ics = {}
    for d in np.unique(dates):
        mask = dates == d
        if mask.sum() < 5:
            continue
        rho, _ = spearmanr(yt[mask], yp[mask])
        if np.isfinite(rho):
            daily_ics[d] = rho
    if not daily_ics:
        return np.nan, np.nan, pd.Series(dtype=float)
    s = pd.Series(daily_ics)
    mean_ic = s.mean()
    std_ic = s.std()
    icir = mean_ic / std_ic if (std_ic and std_ic > 1e-12) else np.nan
    return float(mean_ic), float(icir), s


# ────────────────────────────────────────────────────────────────────────────────
# Simple PyTorch models
# ────────────────────────────────────────────────────────────────────────────────


class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_sizes):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_sizes, n_classes=3):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_loaders(X_train, y_train, X_val, y_val, batch_size=2048):
    X_tr = torch.from_numpy(X_train.astype(np.float32))
    X_va = torch.from_numpy(X_val.astype(np.float32))

    if y_train is None:
        return None, None

    if y_train.dtype.kind in "iu":  # classification
        y_tr = torch.from_numpy(y_train.astype(np.int64))
        y_va = torch.from_numpy(y_val.astype(np.int64))
    else:
        y_tr = torch.from_numpy(y_train.astype(np.float32))
        y_va = torch.from_numpy(y_val.astype(np.float32))

    tr_ds = TensorDataset(X_tr, y_tr)
    va_ds = TensorDataset(X_va, y_va)
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False)
    return tr_loader, va_loader


def train_regressor(model, train_loader, val_loader, n_epochs=15, lr=1e-3):
    model.to(DEVICE)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_state = None
    best_val = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optim.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optim.step()

        # simple val MSE
        model.eval()
        se, n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                preds = model(xb)
                se += torch.sum((preds - yb) ** 2).item()
                n += yb.numel()
        val_mse = se / max(n, 1)
        if val_mse < best_val:
            best_val = val_mse
            best_state = model.state_dict()

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def train_classifier(model, train_loader, val_loader, n_epochs=15, lr=1e-3):
    model.to(DEVICE)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_state = None
    best_val = float("inf")

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optim.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optim.step()

        # simple val loss
        model.eval()
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                logits = model(xb)
                loss = loss_fn(logits, yb)
                total_loss += loss.item() * yb.size(0)
                n += yb.size(0)
        val_loss = total_loss / max(n, 1)
        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict()

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def run_gpu_benchmark():
    print("=" * 72)
    print(f"GPU multi-model benchmark (device={DEVICE})")
    print("=" * 72)

    # 1. Load meta
    print("\n[1/4] Loading returns, regime, correlation, cluster/sector ...")
    ret, regime_ser, C, cluster, sector_map, tickers, dates = load_returns_and_meta()
    print(f"  Tickers: {len(tickers)},  Dates: {len(dates)}")

    # 2. Build feature panel
    print("\n[2/4] Building feature panel ...")
    panel = build_feature_panel(tickers, dates)
    target_cols = ["LogReturn_Next", "Direction", "Volatility_Next"]
    feature_cols = [
        c for c in get_feature_cols(panel) if c in panel.columns and c not in ("date", "ticker")
    ]
    if not feature_cols:
        feature_cols = [c for c in panel.columns if c not in ["date", "ticker"] + target_cols]
    n_feat = len(feature_cols)
    print(f"  Features: {n_feat},  Panel rows: {len(panel)}")

    print("\n[3/4] Applying z-score & encoding sector ...")
    panel = zscore_cross_sectional(panel, feature_cols)
    sector_enc = encode_sector(sector_map, tickers)
    panel_dates = sorted(panel["date"].unique())
    regime_ser = regime_ser.reindex(panel_dates).ffill().bfill().fillna(1).astype(int)

    print("\n[3/4] Making train / val / test splits ...")
    train_split, val_split, test_split = make_splits(panel, feature_cols, regime_ser, sector_enc)

    X_tr, yr_tr, yd_tr, yv_tr, _ = train_split
    X_va, yr_va, yd_va, yv_va, _ = val_split
    X_te, yr_te, yd_te, yv_te, idx_te = test_split

    X_train_full = np.vstack([X_tr, X_va])
    yr_train_full = np.concatenate([yr_tr, yr_va]) if yr_tr is not None else None
    yd_train_full = np.concatenate([yd_tr, yd_va]) if yd_tr is not None else None
    yv_train_full = np.concatenate([yv_tr, yv_va]) if yv_tr is not None else None

    print(f"  X_train_full: {X_train_full.shape}, X_test: {X_te.shape}")

    rows = []

    def log_and_append(model_name, target, metrics_dict, extra=None):
        row = {
            "model_type": "GPU-MLP",
            "model_name": model_name,
            "target": target,
        }
        if extra:
            row.update(extra)
        if metrics_dict:
            row.update(metrics_dict)
        rows.append(row)

    # Models: a few small/medium MLPs for each target
    input_dim = X_train_full.shape[1]

    print("\n[4/4] Training GPU models ...")

    # Return regression
    if yr_train_full is not None:
        for name, hidden in [
            ("MLPReg_ret_small", (128, 64)),
            ("MLPReg_ret_medium", (256, 128, 64)),
        ]:
            try:
                print(f"  [GPU-REG] {name} on LogReturn_Next ...")
                tr_loader, va_loader = make_loaders(
                    X_train_full, yr_train_full, X_va, yr_va, batch_size=4096
                )
                model = MLPRegressor(input_dim, hidden)
                model = train_regressor(model, tr_loader, va_loader, n_epochs=20, lr=1e-3)
                model.eval()
                with torch.no_grad():
                    X_te_t = torch.from_numpy(X_te.astype(np.float32)).to(DEVICE)
                    preds = model(X_te_t).cpu().numpy()
                ic, icir, _ = daily_rank_ic(idx_te.get_level_values(0), yr_te, preds)
                met = regression_metrics(yr_te, preds)
                met.update({"return_ic": ic, "return_icir": icir})
                log_and_append(name, "LogReturn_Next", met)
            except Exception as e:
                log_and_append(name, "LogReturn_Next", {}, {"error": str(e)})

    # Volatility regression
    if yv_train_full is not None:
        for name, hidden in [
            ("MLPReg_vol_small", (128, 64)),
            ("MLPReg_vol_medium", (256, 128, 64)),
        ]:
            try:
                print(f"  [GPU-REG] {name} on Volatility_Next ...")
                tr_loader, va_loader = make_loaders(
                    X_train_full, yv_train_full, X_va, yv_va, batch_size=4096
                )
                model = MLPRegressor(input_dim, hidden)
                model = train_regressor(model, tr_loader, va_loader, n_epochs=20, lr=1e-3)
                model.eval()
                with torch.no_grad():
                    X_te_t = torch.from_numpy(X_te.astype(np.float32)).to(DEVICE)
                    preds = model(X_te_t).cpu().numpy()
                met = regression_metrics(yv_te, preds)
                log_and_append(name, "Volatility_Next", met)
            except Exception as e:
                log_and_append(name, "Volatility_Next", {}, {"error": str(e)})

    # Direction classification (3-class)
    if yd_train_full is not None:
        yd_tr_full_int = yd_train_full.astype(int)
        for name, hidden in [
            ("MLPClf_dir_small", (128, 64)),
            ("MLPClf_dir_medium", (256, 128, 64)),
        ]:
            try:
                print(f"  [GPU-CLF] {name} on Direction ...")
                tr_loader, va_loader = make_loaders(
                    X_train_full, yd_tr_full_int, X_va, yd_va, batch_size=4096
                )
                model = MLPClassifier(input_dim, hidden, n_classes=3)
                model = train_classifier(model, tr_loader, va_loader, n_epochs=20, lr=1e-3)
                model.eval()
                with torch.no_grad():
                    X_te_t = torch.from_numpy(X_te.astype(np.float32)).to(DEVICE)
                    logits = model(X_te_t).cpu().numpy()
                    preds = np.argmax(logits, axis=1)
                met = classification_metrics(yd_te, preds)
                log_and_append(name, "Direction", met)
            except Exception as e:
                log_and_append(name, "Direction", {}, {"error": str(e)})

    df = pd.DataFrame(rows)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = os.path.join(RESULTS_DIR, f"gpu_multi_model_results_{timestamp}.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nAll GPU model metrics saved to: {csv_path}")

    # Simple performance analysis plots (saved as PNGs, paths printed to console)
    def plot_bar(df_sub, metric, title, fname):
        if metric not in df_sub.columns:
            return
        dfm = df_sub.dropna(subset=[metric]).sort_values(metric, ascending=False)
        if dfm.empty:
            return
        top = dfm.head(10)
        plt.figure(figsize=(8, 4))
        plt.barh(top["model_name"], top[metric])
        plt.xlabel(metric)
        plt.title(title)
        plt.gca().invert_yaxis()
        plt.tight_layout()
        out_path = os.path.join(RESULTS_DIR, fname)
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved plot: {out_path}")

    # Return IC / ICIR
    df_ret = df[df["target"] == "LogReturn_Next"]
    plot_bar(df_ret, "return_ic", "Return models — IC (GPU-MLP)", "gpu_ret_ic.png")
    plot_bar(df_ret, "return_icir", "Return models — ICIR (GPU-MLP)", "gpu_ret_icir.png")

    # Direction accuracy
    df_dir = df[df["target"] == "Direction"]
    plot_bar(df_dir, "accuracy", "Direction models — accuracy (GPU-MLP)", "gpu_dir_acc.png")
    plot_bar(df_dir, "auc", "Direction models — AUC (GPU-MLP)", "gpu_dir_auc.png")

    # Vol MSE (lower is better, flip sign so higher bar = better)
    df_vol = df[df["target"] == "Volatility_Next"].copy()
    if not df_vol.empty and "mse" in df_vol.columns:
        df_vol["neg_mse"] = -df_vol["mse"]
        plot_bar(df_vol, "neg_mse", "Volatility models — negative MSE (GPU-MLP)", "gpu_vol_mse.png")

    # Print a compact text summary in console
    print("\nGPU model performance summary:")
    for target, metric, higher_better in [
        ("LogReturn_Next", "return_ic", True),
        ("Direction", "accuracy", True),
        ("Volatility_Next", "mse", False),
    ]:
        if metric not in df.columns:
            continue
        df_t = df[df["target"] == target].dropna(subset=[metric])
        if df_t.empty:
            continue
        best_row = df_t.sort_values(metric, ascending=not higher_better).iloc[0]
        print(
            f"  Target={target:16s}  Best={best_row['model_name']:20s}  "
            f"{metric}={best_row[metric]:.4f}"
        )


if __name__ == "__main__":
    run_gpu_benchmark()



GPU multi-model benchmark (device=cuda)

[1/4] Loading returns, regime, correlation, cluster/sector ...
  returns_matrix.csv not found; building from /content/finance_output ...


FileNotFoundError: Missing: /content/correlation_analysis/returns_matrix.csv and could not build from /content/finance_output (no *_historical.csv with Close).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║        GPU FINANCE BENCHMARK — SINGLE CELL, COLAB READY                    ║
# ║  Just run this entire cell. It installs deps, downloads data, and trains.  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── STEP 0: Install dependencies ─────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "yfinance", "numpy", "pandas", "scikit-learn",
                       "scipy", "torch", "matplotlib", "-q"])

# ── STEP 1: Imports ───────────────────────────────────────────────────────────
import os
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    accuracy_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr

# ── STEP 2: Configuration ─────────────────────────────────────────────────────
TICKERS = [
    "AAPL","MSFT","GOOGL","AMZN","META","NVDA","TSLA","JPM","V","JNJ",
    "UNH","XOM","PG","MA","HD","CVX","MRK","ABBV","PEP","KO",
    "BAC","WMT","LLY","AVGO","TMO","COST","DIS","ADBE","CRM","NFLX",
]

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR   = "/content" if os.path.isdir("/content") else os.getcwd()
DATA_DIR   = os.path.join(BASE_DIR, "finance_output")
CORR_DIR   = os.path.join(BASE_DIR, "correlation_analysis")
OUT_DIR    = os.path.join(BASE_DIR, "comparison_report")
RESULTS_DIR = os.path.join(OUT_DIR, "gpu_multi_model_benchmark")

for d in [DATA_DIR, CORR_DIR, OUT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

TRAIN_END = "2024-06-30"
VAL_END   = "2025-03-31"
TEST_END  = "2026-03-31"

TARGET_COLS = ["LogReturn_Next", "Direction", "Volatility_Next"]

print(f"Device : {DEVICE}")
print(f"Base   : {BASE_DIR}")

# ── STEP 3: Download historical data ─────────────────────────────────────────
print("\n[DATA] Downloading historical OHLCV data from Yahoo Finance ...")
closes = {}
valid_tickers = []

for ticker in TICKERS:
    path = os.path.join(DATA_DIR, f"{ticker}_historical.csv")
    try:
        if os.path.isfile(path):
            df = pd.read_csv(path, index_col=0)
            df.index = pd.to_datetime(df.index, errors="coerce")
            df = df[~df.index.isna()].sort_index()
        else:
            df = yf.download(ticker, start="2015-01-01", auto_adjust=True, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.index = pd.to_datetime(df.index)
            df.to_csv(path)

        if "Close" not in df.columns or len(df) < 300:
            print(f"  Skipped {ticker} (insufficient data)")
            continue

        closes[ticker] = df["Close"].astype(float)
        valid_tickers.append(ticker)
        print(f"  ✓ {ticker}  ({len(df)} rows)")
    except Exception as ex:
        print(f"  ✗ {ticker}: {ex}")

if not closes:
    raise RuntimeError("No ticker data downloaded. Check your internet connection.")

# ── STEP 4: Build returns_matrix.csv ─────────────────────────────────────────
print("\n[DATA] Building returns_matrix.csv ...")
close_df = pd.DataFrame(closes).sort_index()
ret_matrix = np.log(close_df / close_df.shift(1))
ret_path = os.path.join(CORR_DIR, "returns_matrix.csv")
ret_matrix.to_csv(ret_path)
print(f"  Saved -> {ret_path}  shape={ret_matrix.shape}")

# ── STEP 5: Build regime labels (default = 'medium') ─────────────────────────
regime_path = os.path.join(CORR_DIR, "phase5_regime_labels.csv")
pd.Series(
    ["medium"] * len(ret_matrix.index),
    index=ret_matrix.index,
    name="rolling_mean_r",
).to_csv(regime_path)
print(f"  Saved -> {regime_path}")

# ═════════════════════════════════════════════════════════════════════════════
# CORE FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════

def engineer_features(raw: pd.DataFrame, direction_threshold: float = 0.005) -> pd.DataFrame:
    df = raw.copy()
    if "Close" not in df.columns:
        return pd.DataFrame()
    df = df.sort_index()
    c = df["Close"].astype(float)
    logret = np.log(c / c.shift(1))
    out = pd.DataFrame(index=df.index)

    # Features — no lookahead
    out["Ret1"]   = logret.shift(1)
    out["Ret2"]   = logret.shift(2)
    out["Ret5"]   = logret.shift(5)
    out["Mom10"]  = (c / c.shift(10) - 1.0)
    out["Mom20"]  = (c / c.shift(20) - 1.0)
    out["Vol5"]   = logret.rolling(5).std().shift(1)
    out["Vol20"]  = logret.rolling(20).std().shift(1)
    out["Range1"] = (
        ((df["High"].astype(float) - df["Low"].astype(float)) / c).shift(1)
        if "High" in df.columns and "Low" in df.columns else np.nan
    )
    out["VolChg"] = (out["Vol5"] / (out["Vol20"] + 1e-12) - 1.0)

    # Targets — forward-only
    out["LogReturn_Next"]  = logret.shift(-1)
    out["Volatility_Next"] = logret.shift(-1).rolling(5).std()
    nxt = out["LogReturn_Next"]
    out["Direction"] = np.where(
        nxt >  direction_threshold,  1,
        np.where(nxt < -direction_threshold, -1, 0),
    )
    out = out.dropna(subset=["LogReturn_Next", "Volatility_Next"])
    return out


def get_feature_cols(df: pd.DataFrame) -> list:
    ignore = set(TARGET_COLS)
    return [c for c in df.columns if c not in ignore]


def regression_metrics(y_true, y_pred):
    if y_true is None or y_pred is None:
        return {}
    yt = np.asarray(y_true).ravel().astype(float)
    yp = np.asarray(y_pred).ravel().astype(float)
    return {
        "mse": float(mean_squared_error(yt, yp)),
        "mae": float(mean_absolute_error(yt, yp)),
    }


def classification_metrics(y_true, y_pred):
    if y_true is None or y_pred is None:
        return {}
    yt = np.asarray(y_true).ravel().astype(int)
    yp = np.asarray(y_pred).ravel().astype(int)
    try:
        auc = roc_auc_score(yt, yp) if len(np.unique(yt)) == 2 else 0.5
    except Exception:
        auc = 0.5
    return {
        "accuracy":  float(accuracy_score(yt, yp)),
        "precision": float(precision_score(yt, yp, average="macro", zero_division=0)),
        "recall":    float(recall_score(yt, yp, average="macro", zero_division=0)),
        "auc":       float(auc),
    }


def load_returns_and_meta():
    ret = pd.read_csv(ret_path, index_col=0)
    ret.index = pd.to_datetime(ret.index, errors="coerce")
    ret = ret[~ret.index.isna()].sort_index()
    tickers = list(ret.columns)
    dates   = ret.index

    regime_raw = pd.read_csv(regime_path, index_col=0, parse_dates=True).squeeze()
    if hasattr(regime_raw, "columns") and "rolling_mean_r" in regime_raw.columns:
        regime_raw = regime_raw["rolling_mean_r"]
    regime_raw = regime_raw.reindex(dates).ffill().bfill()
    regime_map = {"low": 0, "medium": 1, "high": 2}
    regime_int = regime_raw.map(regime_map).fillna(1).astype(int)

    corr_path = os.path.join(CORR_DIR, "correlation_matrix_full.csv")
    C = pd.read_csv(corr_path, index_col=0) if os.path.isfile(corr_path) else ret.corr()
    C = C.reindex(index=tickers, columns=tickers).fillna(0)

    cluster_path = os.path.join(CORR_DIR, "phase3_cluster_membership.csv")
    cluster, sector = {}, {}
    if os.path.isfile(cluster_path):
        cm = pd.read_csv(cluster_path)
        for _, row in cm.iterrows():
            t = row.get("ticker")
            if t in tickers:
                cluster[t] = int(row.get("empirical_cluster", 0))
                sector[t]  = str(row.get("GICS_sector", "Other"))
    for t in tickers:
        if t not in sector:  sector[t]  = "Other"
        if t not in cluster: cluster[t] = 0

    return ret, regime_int, C, cluster, sector, tickers, dates


def build_feature_panel(tickers, dates) -> pd.DataFrame:
    rows = []
    skipped = 0
    dates = pd.DatetimeIndex(pd.to_datetime(pd.Index(dates), errors="coerce")).dropna().sort_values()
    date_min, date_max = dates.min(), dates.max()
    dates_set = set(dates.to_pydatetime())

    for i, sym in enumerate(tickers):
        if (i + 1) % 10 == 0:
            print(f"  Building features {i+1}/{len(tickers)} ...")
        path = os.path.join(DATA_DIR, f"{sym}_historical.csv")
        if not os.path.isfile(path):
            skipped += 1
            continue
        try:
            raw = pd.read_csv(path, index_col=0)
            raw.index = pd.to_datetime(raw.index, errors="coerce")
            raw = raw[~raw.index.isna()].sort_index()
            if "Close" not in raw.columns or len(raw) < 300:
                skipped += 1
                continue
            df_sym = engineer_features(raw)
            fc = get_feature_cols(df_sym)
            if not fc or "LogReturn_Next" not in df_sym.columns:
                skipped += 1
                continue
            df_sym = df_sym.loc[
                (df_sym.index >= date_min - pd.Timedelta(days=400))
                & (df_sym.index <= date_max + pd.Timedelta(days=5))
            ]
            for ts in df_sym.index:
                if ts.to_pydatetime() not in dates_set:
                    continue
                row = {"date": ts, "ticker": sym}
                for c in fc:
                    if c in df_sym.columns:
                        row[c] = df_sym.loc[ts, c]
                for tgt in TARGET_COLS:
                    if tgt in df_sym.columns:
                        row[tgt] = df_sym.loc[ts, tgt]
                if len(row) > 5:
                    rows.append(row)
        except Exception:
            skipped += 1

    if skipped:
        print(f"  [build_panel] skipped {skipped} tickers")
    if not rows:
        raise RuntimeError("No feature rows built. Check finance_output CSVs.")
    return pd.DataFrame(rows)


def zscore_cross_sectional(panel: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    out = panel.copy()
    for _, grp in out.groupby("date"):
        for c in feature_cols:
            if c not in grp.columns:
                continue
            s = grp[c].astype(float)
            mu, std = s.mean(), s.std()
            out.loc[grp.index, c] = (s - mu) / std if (std and std > 1e-12) else 0.0
    return out


def encode_sector(sector_map: dict, tickers: list) -> dict:
    le = LabelEncoder()
    labels = [sector_map.get(t, "Other") for t in tickers]
    le.fit(labels)
    return {t: int(le.transform([sector_map.get(t, "Other")])[0]) for t in tickers}


def make_splits(panel, feature_cols, regime_ser, sector_enc):
    p = panel.set_index(["date", "ticker"])
    all_dates = sorted(p.index.get_level_values(0).unique())

    train_dates = [d for d in all_dates if d <= pd.Timestamp(TRAIN_END)]
    val_dates   = [d for d in all_dates if pd.Timestamp(TRAIN_END) < d <= pd.Timestamp(VAL_END)]
    test_dates  = [d for d in all_dates if pd.Timestamp(VAL_END)   < d <= pd.Timestamp(TEST_END)]

    def get_split(date_list):
        mask = p.index.get_level_values(0).isin(date_list)
        P = p.loc[mask].copy().dropna(subset=feature_cols, how="all")
        X   = P[feature_cols].fillna(0).values.astype(np.float32)
        sec = np.array([sector_enc.get(t, 0) for t in P.index.get_level_values(1)],
                       dtype=np.float32).reshape(-1, 1)
        reg = regime_ser.reindex(P.index.get_level_values(0)).values.reshape(-1, 1).astype(np.float32)
        X   = np.hstack([X, reg, sec])
        y_ret = P["LogReturn_Next"].values  if "LogReturn_Next"  in P.columns else None
        y_dir = P["Direction"].values       if "Direction"       in P.columns else None
        y_vol = P["Volatility_Next"].values if "Volatility_Next" in P.columns else None
        return X, y_ret, y_dir, y_vol, P.index

    return get_split(train_dates), get_split(val_dates), get_split(test_dates)


def daily_rank_ic(dates_arr, y_true, y_pred):
    if y_true is None or y_pred is None:
        return np.nan, np.nan, pd.Series(dtype=float)
    dates, yt, yp = np.asarray(dates_arr), np.asarray(y_true), np.asarray(y_pred)
    daily_ics = {}
    for d in np.unique(dates):
        mask = dates == d
        if mask.sum() < 5:
            continue
        rho, _ = spearmanr(yt[mask], yp[mask])
        if np.isfinite(rho):
            daily_ics[d] = rho
    if not daily_ics:
        return np.nan, np.nan, pd.Series(dtype=float)
    s = pd.Series(daily_ics)
    mean_ic, std_ic = s.mean(), s.std()
    icir = mean_ic / std_ic if (std_ic and std_ic > 1e-12) else np.nan
    return float(mean_ic), float(icir), s


# ── PyTorch Models ────────────────────────────────────────────────────────────

class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_sizes):
        super().__init__()
        layers, last = [], input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        layers.append(nn.Linear(last, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_sizes, n_classes=3):
        super().__init__()
        layers, last = [], input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        layers.append(nn.Linear(last, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_loaders(X_train, y_train, X_val, y_val, batch_size=2048):
    X_tr = torch.from_numpy(X_train.astype(np.float32))
    X_va = torch.from_numpy(X_val.astype(np.float32))
    if y_train is None:
        return None, None
    # FIX: Direction labels are {-1,0,1} — remap to {0,1,2} for CrossEntropyLoss
    if y_train.dtype.kind in "fi" and np.any(y_train < 0):
        y_train = y_train + 1   # -1→0, 0→1, 1→2
        y_val   = y_val   + 1
    if y_train.dtype.kind in "iu" or np.issubdtype(y_train.dtype, np.integer):
        y_tr = torch.from_numpy(y_train.astype(np.int64))
        y_va = torch.from_numpy(y_val.astype(np.int64))
    else:
        y_tr = torch.from_numpy(y_train.astype(np.float32))
        y_va = torch.from_numpy(y_val.astype(np.float32))
    tr_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size, shuffle=False)
    return tr_loader, va_loader


def train_regressor(model, train_loader, val_loader, n_epochs=20, lr=1e-3):
    model.to(DEVICE)
    optim   = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_state, best_val = None, float("inf")
    for _ in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optim.zero_grad()
            loss_fn(model(xb), yb).backward()
            optim.step()
        model.eval()
        se, n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                se += torch.sum((model(xb) - yb) ** 2).item()
                n  += yb.numel()
        val_mse = se / max(n, 1)
        if val_mse < best_val:
            best_val   = val_mse
            best_state = model.state_dict()
    if best_state:
        model.load_state_dict(best_state)
    return model


def train_classifier(model, train_loader, val_loader, n_epochs=20, lr=1e-3):
    model.to(DEVICE)
    optim   = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_state, best_val = None, float("inf")
    for _ in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optim.zero_grad()
            loss_fn(model(xb), yb).backward()
            optim.step()
        model.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                total += loss_fn(model(xb), yb).item() * yb.size(0)
                n     += yb.size(0)
        val_loss = total / max(n, 1)
        if val_loss < best_val:
            best_val   = val_loss
            best_state = model.state_dict()
    if best_state:
        model.load_state_dict(best_state)
    return model


# ═════════════════════════════════════════════════════════════════════════════
# MAIN BENCHMARK
# ═════════════════════════════════════════════════════════════════════════════

def run_gpu_benchmark():
    print("=" * 72)
    print(f"GPU multi-model benchmark  (device={DEVICE})")
    print("=" * 72)

    # 1. Load meta
    print("\n[1/4] Loading returns, regime, correlation, cluster/sector ...")
    ret, regime_ser, C, cluster, sector_map, tickers, dates = load_returns_and_meta()
    print(f"  Tickers: {len(tickers)},  Dates: {len(dates)}")

    # 2. Build feature panel
    print("\n[2/4] Building feature panel ...")
    panel = build_feature_panel(tickers, dates)
    feature_cols = [
        c for c in get_feature_cols(panel)
        if c in panel.columns and c not in ("date", "ticker")
    ]
    if not feature_cols:
        feature_cols = [c for c in panel.columns if c not in ["date", "ticker"] + TARGET_COLS]
    print(f"  Features: {len(feature_cols)},  Panel rows: {len(panel)}")

    # 3. Z-score + encode
    print("\n[3/4] Applying z-score & encoding sector ...")
    panel       = zscore_cross_sectional(panel, feature_cols)
    sector_enc  = encode_sector(sector_map, tickers)
    panel_dates = sorted(panel["date"].unique())
    regime_ser  = regime_ser.reindex(panel_dates).ffill().bfill().fillna(1).astype(int)

    print("[3/4] Making train / val / test splits ...")
    train_split, val_split, test_split = make_splits(panel, feature_cols, regime_ser, sector_enc)

    X_tr, yr_tr, yd_tr, yv_tr, _      = train_split
    X_va, yr_va, yd_va, yv_va, _      = val_split
    X_te, yr_te, yd_te, yv_te, idx_te = test_split

    X_full  = np.vstack([X_tr, X_va])
    yr_full = np.concatenate([yr_tr, yr_va]) if yr_tr is not None else None
    yd_full = np.concatenate([yd_tr, yd_va]) if yd_tr is not None else None
    yv_full = np.concatenate([yv_tr, yv_va]) if yv_tr is not None else None

    print(f"  Train: {X_full.shape},  Test: {X_te.shape}")

    input_dim = X_full.shape[1]
    rows = []

    def record(model_name, target, metrics, extra=None):
        row = {"model_type": "GPU-MLP", "model_name": model_name, "target": target}
        if extra:   row.update(extra)
        if metrics: row.update(metrics)
        rows.append(row)

    # 4. Train models
    print("\n[4/4] Training GPU models ...")

    # — Return regression —
    if yr_full is not None:
        for name, hidden in [("MLPReg_ret_small", (128, 64)),
                              ("MLPReg_ret_medium", (256, 128, 64))]:
            try:
                print(f"  [REG] {name} → LogReturn_Next")
                tr_l, va_l = make_loaders(X_full, yr_full, X_va, yr_va, 4096)
                mdl = train_regressor(MLPRegressor(input_dim, hidden), tr_l, va_l)
                mdl.eval()
                with torch.no_grad():
                    preds = mdl(torch.from_numpy(X_te).to(DEVICE)).cpu().numpy()
                ic, icir, _ = daily_rank_ic(idx_te.get_level_values(0), yr_te, preds)
                met = regression_metrics(yr_te, preds)
                met.update({"return_ic": ic, "return_icir": icir})
                record(name, "LogReturn_Next", met)
                print(f"    MSE={met['mse']:.6f}  IC={ic:.4f}  ICIR={icir:.4f}")
            except Exception as e:
                record(name, "LogReturn_Next", {}, {"error": str(e)})
                print(f"    ERROR: {e}")

    # — Volatility regression —
    if yv_full is not None:
        for name, hidden in [("MLPReg_vol_small", (128, 64)),
                              ("MLPReg_vol_medium", (256, 128, 64))]:
            try:
                print(f"  [REG] {name} → Volatility_Next")
                tr_l, va_l = make_loaders(X_full, yv_full, X_va, yv_va, 4096)
                mdl = train_regressor(MLPRegressor(input_dim, hidden), tr_l, va_l)
                mdl.eval()
                with torch.no_grad():
                    preds = mdl(torch.from_numpy(X_te).to(DEVICE)).cpu().numpy()
                met = regression_metrics(yv_te, preds)
                record(name, "Volatility_Next", met)
                print(f"    MSE={met['mse']:.6f}  MAE={met['mae']:.6f}")
            except Exception as e:
                record(name, "Volatility_Next", {}, {"error": str(e)})
                print(f"    ERROR: {e}")

    # — Direction classification (FIX: remap {-1,0,1} → {0,1,2}) —
    if yd_full is not None:
        yd_full_int = yd_full.astype(int) + 1   # remap: -1→0, 0→1, 1→2
        yd_va_int   = yd_va.astype(int)   + 1
        yd_te_int   = yd_te.astype(int)   + 1

        for name, hidden in [("MLPClf_dir_small", (128, 64)),
                              ("MLPClf_dir_medium", (256, 128, 64))]:
            try:
                print(f"  [CLF] {name} → Direction")
                tr_l, va_l = make_loaders(X_full, yd_full_int, X_va, yd_va_int, 4096)
                mdl = train_classifier(MLPClassifier(input_dim, hidden, n_classes=3), tr_l, va_l)
                mdl.eval()
                with torch.no_grad():
                    logits = mdl(torch.from_numpy(X_te).to(DEVICE)).cpu().numpy()
                    preds  = np.argmax(logits, axis=1)
                met = classification_metrics(yd_te_int, preds)
                record(name, "Direction", met)
                print(f"    Acc={met['accuracy']:.4f}  Prec={met['precision']:.4f}  AUC={met['auc']:.4f}")
            except Exception as e:
                record(name, "Direction", {}, {"error": str(e)})
                print(f"    ERROR: {e}")

    # ── Save results CSV ──────────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    ts  = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = os.path.join(RESULTS_DIR, f"gpu_results_{ts}.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nResults saved → {csv_path}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    def plot_bar(df_sub, metric, title, fname):
        if metric not in df_sub.columns:
            return
        dfm = df_sub.dropna(subset=[metric]).sort_values(metric, ascending=False)
        if dfm.empty:
            return
        top = dfm.head(10)
        plt.figure(figsize=(8, 4))
        plt.barh(top["model_name"], top[metric])
        plt.xlabel(metric)
        plt.title(title)
        plt.gca().invert_yaxis()
        plt.tight_layout()
        out_path = os.path.join(RESULTS_DIR, fname)
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Plot saved → {out_path}")

    df_ret = df[df["target"] == "LogReturn_Next"]
    plot_bar(df_ret, "return_ic",   "Return IC (GPU-MLP)",   "gpu_ret_ic.png")
    plot_bar(df_ret, "return_icir", "Return ICIR (GPU-MLP)", "gpu_ret_icir.png")

    df_dir = df[df["target"] == "Direction"]
    plot_bar(df_dir, "accuracy", "Direction Accuracy (GPU-MLP)", "gpu_dir_acc.png")
    plot_bar(df_dir, "auc",      "Direction AUC (GPU-MLP)",      "gpu_dir_auc.png")

    df_vol = df[df["target"] == "Volatility_Next"].copy()
    if not df_vol.empty and "mse" in df_vol.columns:
        df_vol["neg_mse"] = -df_vol["mse"]
        plot_bar(df_vol, "neg_mse", "Volatility −MSE (GPU-MLP)", "gpu_vol_mse.png")

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("PERFORMANCE SUMMARY")
    print("=" * 72)
    for target, metric, higher_better in [
        ("LogReturn_Next",  "return_ic", True),
        ("Direction",       "accuracy",  True),
        ("Volatility_Next", "mse",       False),
    ]:
        if metric not in df.columns:
            continue
        df_t = df[df["target"] == target].dropna(subset=[metric])
        if df_t.empty:
            continue
        best = df_t.sort_values(metric, ascending=not higher_better).iloc[0]
        print(f"  {target:<18s}  Best: {best['model_name']:<22s}  {metric}={best[metric]:.4f}")

    print("\nDone! ✓")


# ── Run ───────────────────────────────────────────────────────────────────────
run_gpu_benchmark()